# Spanish Handwritten OCR Classifier — v2
**107-class character recognition | EfficientNet-V2-S + ArcFace**

Cambios v2:
- Fix: EMNIST escribe en /kaggle/working/ (filesystem de solo lectura)
- Fix: indexación EMNIST usa `ds.targets` directamente
- Fix: verack busca en múltiples rutas con NFC normalization
- Fix: sueiras maneja .tar.gz completo + partes
- Fix: DataFrame vacío manejado con schema explícito
- Fix: val_loss evaluada sin ArcFace margin (cosine similarity directa)
- Mejora: dropout 0.3→0.5, label_smoothing=0.1, weight_decay→5e-4
- Mejora: resize con padding (aspect-ratio preservado) para info de tamaño
- Mejora: freeze_epochs=2 con lr head elevado
- Mejora: scheduler T_0=20 para menos disrupciones
- Mejora: ArcFace m=0.2
- Mejora: augmentación más agresiva (Perspective, morfológica, GridDistortion)
- Mejora: Mixup augmentation en training loop
- Mejora: TTA (Test-Time Augmentation) en evaluación final
- Mejora: lr diferenciado por capa (capas tempranas × 0.1, tardías × 0.5)

## Cell 0 — Instalación de dependencias

In [ ]:
!pip install timm albumentations opencv-python-headless onnx onnxscript onnxruntime pyarrow -q

## Cell 1 — Imports y configuración global

In [ ]:
import os, sys, json, time, math, random, shutil, tarfile, unicodedata
import glob, warnings, subprocess
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import torchvision
import torchvision.transforms as transforms
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import onnx
import onnxruntime as ort

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_BASE = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

CHAR_MAP_PATH = INPUT_BASE / 'datasets/danielperegrinoperez/char-map/char_map.json'

EMNIST_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/crawford/emnist',
    INPUT_BASE / 'emnist',
    INPUT_BASE / 'crawford-emnist',
]
VERACK_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/verack/spanish-handwritten-characterswords',
    INPUT_BASE / 'verack/spanish-handwritten-characterswords',
    INPUT_BASE / 'spanish-handwritten-characterswords',
]
SUEIRAS_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/sueiras/handwritting-characters-database',
    INPUT_BASE / 'sueiras/handwritting-characters-database',
    INPUT_BASE / 'handwritting-characters-database',
]

# Directorios de trabajo
EMNIST_WORK_DIR     = WORK / 'emnist_data'
EMNIST_EXPORT_DIR   = WORK / 'emnist_images'
SUEIRAS_EXTRACT_DIR = WORK / 'sueiras_extracted'
SYNTH_DIR           = WORK / 'synthetic_chars'
ACCENT_AUG_DIR      = WORK / 'accent_augmented'   # NUEVO: acentuadas desde EMNIST
HW_FONTS_DIR        = WORK / 'handwriting_fonts'   # NUEVO: fuentes manuscritas

for d in [EMNIST_WORK_DIR, EMNIST_EXPORT_DIR, SUEIRAS_EXTRACT_DIR,
          SYNTH_DIR, ACCENT_AUG_DIR, HW_FONTS_DIR]:
    d.mkdir(exist_ok=True)

META_PATH      = WORK / 'merged_metadata.parquet'
COVERAGE_PATH  = WORK / 'class_coverage_report.json'
TRAIN_CFG_PATH = WORK / 'train_config.json'
BEST_PT_PATH   = WORK / 'best_classifier.pt'
BEST_ONNX_PATH = WORK / 'best_classifier.onnx'
METRICS_PATH   = WORK / 'metrics_report.json'
CURVES_PATH    = WORK / 'training_curves.png'
CONFMAT_PATH   = WORK / 'confusion_matrix.png'
TOP10_PATH     = WORK / 'top10_confused_pairs.json'

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Run ID: {RUN_ID}')
print(f'PyTorch {torch.__version__}  |  timm {timm.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Hiperparámetros ─────────────────────────────────────────────────────────
IMG_SIZE      = 128
NUM_CLASSES   = 107
BATCH_SIZE    = 64
NUM_WORKERS   = 4

# Learning rates
LR_HEAD       = 5e-3    # CAMBIADO: un poco menor para ArcFace más estable
LR_BACKBONE   = 1e-4    # CAMBIADO: más conservador para fine-tune
WEIGHT_DECAY  = 5e-4

# Entrenamiento
MAX_EPOCHS    = 50       # CAMBIADO: más épocas, early stopping decide
FREEZE_EPOCHS = 5        # CAMBIADO: 2→5 para estabilizar ArcFace primero
PATIENCE      = 12

# ArcFace
ARCFACE_S     = 30.0
ARCFACE_M     = 0.15

# Regularización
DROPOUT_RATE  = 0.4      # CAMBIADO: 0.5→0.4 (0.5 era excesivo)
LABEL_SMOOTH  = 0.05
MIXUP_ALPHA   = 0.2

# TTA
TTA_N         = 5        # CAMBIADO: 4→5

# Datos
EMNIST_MAX_PER_CLASS = 800   # CAMBIADO: 500→800 (más datos reales)
SYNTH_PER_CLASS      = 500   # CAMBIADO: 100→500 (mucha más diversidad)
SYNTH_PER_CLASS_STROKE = 300 # Trazos dibujados
MIN_REAL_SAMPLES     = 100   # CAMBIADO: 50→100

# ── NUEVO: Accent Augmentation Config ────────────────────────────────────────
# Mapeo: carácter base → lista de (carácter acentuado, tipo de acento)
ACCENT_MAP = {
    'a': [('á', 'acute')],
    'e': [('é', 'acute')],
    'i': [('í', 'acute')],
    'o': [('ó', 'acute')],
    'u': [('ú', 'acute'), ('ü', 'dieresis')],
    'n': [('ñ', 'tilde')],
    'A': [('Á', 'acute')],
    'E': [('É', 'acute')],
    'I': [('Í', 'acute')],
    'O': [('Ó', 'acute')],
    'U': [('Ú', 'acute'), ('Ü', 'dieresis')],
    'N': [('Ñ', 'tilde')],
}
ACCENT_SAMPLES_PER_BASE = 400  # Cuántas imágenes generar por par base→acentuada

# ── NUEVO: Pares confusos completos ─────────────────────────────────────────
CONFUSED_PAIRS = [
    # Case confusion
    ('c', 'C'), ('C', 'c'), ('s', 'S'), ('S', 's'),
    ('o', 'O'), ('O', 'o'), ('u', 'U'), ('U', 'u'),
    ('v', 'V'), ('V', 'v'), ('p', 'P'), ('P', 'p'),
    ('f', 'F'), ('F', 'f'), ('w', 'W'), ('W', 'w'),
    ('x', 'X'), ('X', 'x'), ('k', 'K'), ('K', 'k'),
    ('m', 'M'), ('M', 'm'),
    # Shape confusion
    ('1', 'l'), ('l', '1'), ('O', '0'), ('0', 'O'),
    # NUEVO: Accent confusion (el problema principal)
    ('a', 'á'), ('á', 'a'), ('e', 'é'), ('é', 'e'),
    ('i', 'í'), ('í', 'i'), ('o', 'ó'), ('ó', 'o'),
    ('u', 'ú'), ('ú', 'u'), ('u', 'ü'), ('ü', 'u'),
    ('n', 'ñ'), ('ñ', 'n'),
    ('A', 'Á'), ('Á', 'A'), ('E', 'É'), ('É', 'E'),
    ('I', 'Í'), ('Í', 'I'), ('O', 'Ó'), ('Ó', 'O'),
    ('U', 'Ú'), ('Ú', 'U'), ('U', 'Ü'), ('Ü', 'U'),
    ('N', 'Ñ'), ('Ñ', 'N'),
]
EXTRA_PER_CONFUSED = 100  # Menos por par pero más pares
FONT_SUPPLEMENT_PER_CLASS = 200

# ── NUEVO: Timer de entrenamiento ────────────────────────────────────────────
MAX_TRAINING_MINUTES = 110  # Dejar margen para evaluación + export

print(f'\n=== Configuración v5 (mejorada) ===')
print(f'EMNIST_MAX_PER_CLASS: {EMNIST_MAX_PER_CLASS}')
print(f'SYNTH_PER_CLASS: {SYNTH_PER_CLASS}')
print(f'ACCENT_SAMPLES_PER_BASE: {ACCENT_SAMPLES_PER_BASE}')
print(f'FREEZE_EPOCHS: {FREEZE_EPOCHS}')
print(f'MAX_EPOCHS: {MAX_EPOCHS}')
print(f'LR_HEAD: {LR_HEAD}  LR_BACKBONE: {LR_BACKBONE}')
print(f'DROPOUT_RATE: {DROPOUT_RATE}')

## Cell 2 — Carga de char_map

In [ ]:
with open(CHAR_MAP_PATH) as f:
    char_map = json.load(f)

IDX2CHAR: Dict[int, str] = {int(k): v for k, v in char_map['idx2char'].items()}
CHAR2IDX: Dict[str, int] = {v: int(k) for k, v in char_map['idx2char'].items()}

# Índice NFC-normalizado para robusted de matching
CHAR2IDX_NFC: Dict[str, int] = {
    unicodedata.normalize('NFC', k): v for k, v in CHAR2IDX.items()
}

def char_to_idx(ch: str) -> Optional[int]:
    """Busca un carácter en char_map con fallback NFC."""
    if ch in CHAR2IDX:
        return CHAR2IDX[ch]
    nfc = unicodedata.normalize('NFC', ch)
    if nfc in CHAR2IDX_NFC:
        return CHAR2IDX_NFC[nfc]
    return None

assert len(IDX2CHAR) == NUM_CLASSES
print(f'char_map cargado: {NUM_CLASSES} clases')
print('Clases:', list(IDX2CHAR.values()))

## Cell 3 — Mapeo EMNIST

In [ ]:
emnist_chars = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(ord('A'), ord('Z') + 1)]
    + [chr(c) for c in range(ord('a'), ord('z') + 1)]
)

def emnist_label_to_char(label_int: int) -> Optional[str]:
    if 0 <= label_int < len(emnist_chars):
        return emnist_chars[label_int]
    return None

print(f'EMNIST cubre {len(emnist_chars)} clases')

## Cell 4 — Phase A: EMNIST (FIX: filesystem de solo lectura + indexación rápida)

In [ ]:
def find_existing_path(candidates: List[Path]) -> Optional[Path]:
    for p in candidates:
        if p.exists():
            return p
    return None

def fix_emnist_orientation(arr: np.ndarray) -> np.ndarray:
    arr = cv2.transpose(arr)
    arr = cv2.flip(arr, flipCode=1)
    arr = cv2.bitwise_not(arr)
    return arr


def prepare_emnist_root() -> Optional[Path]:
    """
    FIX: torchvision.datasets.EMNIST necesita escribir en su root.
    Copiamos los .gz a EMNIST_WORK_DIR (en /kaggle/working/) y usamos ese
    directorio como root con download=False.
    Fallback: download=True directo a EMNIST_WORK_DIR si no hay fuente.
    """
    src_root = find_existing_path(EMNIST_CANDIDATE_ROOTS)

    # Directorio destino esperado por torchvision
    tv_dir = EMNIST_WORK_DIR / 'EMNIST' / 'raw'
    tv_dir.mkdir(parents=True, exist_ok=True)

    if src_root is not None:
        print(f'  EMNIST fuente encontrada: {src_root}')
        # Copiar archivos .gz al directorio de trabajo
        gz_files = list(src_root.rglob('*.gz'))
        if not gz_files:
            # Quizás ya está extraído en src_root/EMNIST/raw/
            raw_src = src_root / 'EMNIST' / 'raw'
            if raw_src.exists():
                gz_files = list(raw_src.glob('*.gz'))
        for gz in gz_files:
            dst = tv_dir / gz.name
            if not dst.exists():
                shutil.copy2(gz, dst)
        print(f'  Copiados {len(gz_files)} archivos .gz a {tv_dir}')
        return EMNIST_WORK_DIR
    else:
        print('  EMNIST: no encontrado en rutas candidatas — se descargará')
        return EMNIST_WORK_DIR  # torchvision descargará con download=True


def load_emnist_split(emnist_root: Path, split: str) -> List[Dict]:
    records = []
    print(f'  Cargando EMNIST split={split}...')
    try:
        ds = torchvision.datasets.EMNIST(
            root=str(emnist_root),
            split='byclass',
            train=(split == 'train'),
            download=True,                    # download=True siempre; no-op si ya existe
            transform=transforms.ToTensor()  # CRÍTICO: evita TypeError en collate
        )
    except Exception as e:
        print(f'  WARNING: No se pudo cargar EMNIST {split}: {e}')
        return records

    # FIX: usar ds.targets directamente para indexar (evita iterar 800K samples)
    if hasattr(ds, 'targets'):
        targets = ds.targets.numpy() if hasattr(ds.targets, 'numpy') else np.array(ds.targets)
    else:
        # Fallback lento
        targets = np.array([ds[i][1] for i in range(len(ds))])

    indices_by_class: Dict[int, List[int]] = defaultdict(list)
    for i, lbl in enumerate(targets):
        indices_by_class[int(lbl)].append(i)

    for class_idx, idxs in tqdm(indices_by_class.items(),
                                 desc=f'EMNIST {split} export'):
        char = emnist_label_to_char(class_idx)
        if char is None:
            continue
        global_idx = char_to_idx(char)
        if global_idx is None:
            continue

        selected = idxs[:EMNIST_MAX_PER_CLASS]
        cls_dir = EMNIST_EXPORT_DIR / f'cls{class_idx:02d}'
        cls_dir.mkdir(exist_ok=True)

        for i, ds_i in enumerate(selected):
            tensor, _ = ds[ds_i]
            arr = (tensor.squeeze().numpy() * 255).astype(np.uint8)
            arr = fix_emnist_orientation(arr)
            arr_rgb = cv2.cvtColor(
                cv2.resize(arr, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_GRAY2RGB
            )
            img_path = cls_dir / f'{split}_{i:05d}.png'
            cv2.imwrite(str(img_path), arr_rgb)
            records.append({
                'image_path': str(img_path),
                'label': char,
                'class_idx': global_idx,
                'source': 'emnist',
                'width': IMG_SIZE,
                'height': IMG_SIZE,
            })
    return records


print('=== Extrayendo EMNIST ===')
emnist_root = prepare_emnist_root()
emnist_records: List[Dict] = []
for sp in ['train', 'test']:
    emnist_records += load_emnist_split(emnist_root, sp)
print(f'EMNIST records: {len(emnist_records)}')

## Cell 5 — Phase A: verack (FIX: múltiples rutas + NFC normalization)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Phase A: verack OPTIMIZADO (escaneo más rápido)
# ══════════════════════════════════════════════════════════════════════════════

VERACK_WORK_DIR = WORK / 'verack_data'
VERACK_WORK_DIR.mkdir(exist_ok=True)

def find_verack_root() -> Optional[Path]:
    """Busca verack en rutas montadas."""
    all_candidates = list(VERACK_CANDIDATE_ROOTS)
    
    for item in INPUT_BASE.iterdir() if INPUT_BASE.exists() else []:
        if 'spanish' in item.name.lower() and 'handwrit' in item.name.lower():
            all_candidates.append(item)
        if item.is_dir() and item.name == 'datasets':
            for owner in item.iterdir():
                if not owner.is_dir():
                    continue
                for ds in owner.iterdir():
                    if 'spanish' in ds.name.lower() and 'handwrit' in ds.name.lower():
                        all_candidates.append(ds)
    
    for p in all_candidates:
        if p.exists() and any(p.rglob('*')):
            has_images = any(p.rglob('*.png')) or any(p.rglob('*.jpg'))
            has_dirs = any(d.is_dir() for d in p.iterdir() 
                         if d.name not in ('images', 'labels', 'yolo_dataset_final'))
            if has_images or has_dirs:
                print(f'  verack encontrado en: {p}')
                return p
    
    prev = VERACK_WORK_DIR / 'spanish-handwritten-characterswords'
    if prev.exists() and any(prev.rglob('*')):
        return prev
    
    # Intentar descarga solo si no se encuentra
    try:
        import kagglehub
        path = kagglehub.dataset_download('verack/spanish-handwritten-characterswords')
        src = Path(path)
        if src.exists():
            print(f'  ✓ verack descargado: {src}')
            return src
    except Exception as e:
        print(f'  Descarga falló: {e}')
    
    return None


# Reutilizar los mismos slug mappings
VERACK_SLUG2CHAR = {
    **{c: c for c in 'abcdefghijklmnopqrstuvwxyz'},
    **{c.upper(): c.upper() for c in 'abcdefghijklmnopqrstuvwxyz'},
    'a_acento': 'á', 'e_acento': 'é', 'i_acento': 'í',
    'o_acento': 'ó', 'u_acento': 'ú', 'u_dieresis': 'ü',
    'A_acento': 'Á', 'E_acento': 'É', 'I_acento': 'Í',
    'O_acento': 'Ó', 'U_acento': 'Ú', 'U_dieresis': 'Ü',
    'enie': 'ñ', 'Enie': 'Ñ', 'enye': 'ñ', 'Enye': 'Ñ',
    'ñ': 'ñ', 'Ñ': 'Ñ', 'n_tilde': 'ñ', 'N_tilde': 'Ñ',
    **{str(d): str(d) for d in range(10)},
    'punto': '.', 'coma': ',', 'punto_coma': ';', 'dos_puntos': ':',
    'interrogacion_apertura': '¿', 'interrogacion_cierre': '?',
    'exclamacion_apertura': '¡', 'exclamacion_cierre': '!',
    'parentesis_apertura': '(', 'parentesis_cierre': ')',
    'guion': '-', 'guion_bajo': '_', 'comilla': "'", 'comillas': '"',
    'barra': '/', 'arroba': '@', 'numeral': '#', 'dolar': '$',
    'porcentaje': '%', 'ampersand': '&', 'asterisco': '*',
    'mas': '+', 'igual': '=', 'menor': '<', 'mayor': '>',
}


def verack_folder_to_char(folder_name: str) -> Optional[str]:
    idx = char_to_idx(folder_name)
    if idx is not None:
        return IDX2CHAR[idx]
    nfc = unicodedata.normalize('NFC', folder_name)
    idx = char_to_idx(nfc)
    if idx is not None:
        return IDX2CHAR[idx]
    if folder_name in VERACK_SLUG2CHAR:
        ch = VERACK_SLUG2CHAR[folder_name]
        idx = char_to_idx(ch)
        if idx is not None:
            return ch
    if folder_name.lower() in VERACK_SLUG2CHAR:
        ch = VERACK_SLUG2CHAR[folder_name.lower()]
        idx = char_to_idx(ch)
        if idx is not None:
            return ch
    if len(folder_name) == 1:
        idx = char_to_idx(folder_name)
        if idx is not None:
            return folder_name
    return None


def load_verack_fast(root: Path) -> List[Dict]:
    """
    OPTIMIZADO: Escanea solo las subcarpetas directas de profundidad 1-2.
    No hace rglob en 197K archivos.
    """
    records = []
    matched, skipped_folder, skipped_aspect = 0, 0, 0
    
    # Buscar carpetas con imágenes a profundidad 1 y 2 (no más)
    image_dirs = set()
    
    # Profundidad 1
    for sub in root.iterdir():
        if not sub.is_dir():
            continue
        has_img = any(sub.glob('*.png')) or any(sub.glob('*.jpg'))
        if has_img:
            image_dirs.add(sub)
        # Profundidad 2
        for sub2 in sub.iterdir():
            if not sub2.is_dir():
                continue
            has_img2 = any(sub2.glob('*.png')) or any(sub2.glob('*.jpg'))
            if has_img2:
                image_dirs.add(sub2)
    
    print(f'  Carpetas con imágenes: {len(image_dirs)}')
    
    for folder in tqdm(sorted(image_dirs), desc='verack scan'):
        folder_name = folder.name
        
        if folder_name.startswith('.') or folder_name.lower() in (
            'images', 'labels', 'train', 'val', 'test', 'data',
            'dataset', 'words', 'word', 'sentences'
        ):
            continue
        
        char = verack_folder_to_char(folder_name)
        if char is None:
            skipped_folder += 1
            continue
        
        global_idx = char_to_idx(char)
        if global_idx is None:
            skipped_folder += 1
            continue
        
        imgs = [f for f in folder.iterdir()
                if f.is_file() and f.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp')]
        
        for img_path in imgs:
            try:
                im = cv2.imread(str(img_path))
                if im is None:
                    continue
                h, w = im.shape[:2]
                if w > 4 * h:
                    skipped_aspect += 1
                    continue
                records.append({
                    'image_path': str(img_path),
                    'label': char,
                    'class_idx': global_idx,
                    'source': 'verack',
                    'width': w,
                    'height': h,
                })
                matched += 1
            except Exception:
                continue
    
    print(f'  verack: {matched} imgs | {skipped_folder} carpetas skip | '
          f'{skipped_aspect} aspecto skip')
    return records


print('=== Cargando verack Spanish Handwritten ===')
verack_root = find_verack_root()
verack_records: List[Dict] = []
if verack_root is not None:
    verack_records = load_verack_fast(verack_root)
else:
    print('  verack no disponible')
print(f'verack records: {len(verack_records)}')

## Cell 6 — Phase A: sueiras (FIX: tar.gz completo + partes + múltiples rutas)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Phase A: sueiras (SKIP si duplicado + funciones completas)
# ══════════════════════════════════════════════════════════════════════════════

SUEIRAS_WORK_DIR = WORK / 'sueiras_data'
SUEIRAS_WORK_DIR.mkdir(exist_ok=True)


def find_existing_path(candidates: List[Path]) -> Optional[Path]:
    """Busca el primer path que exista en la lista de candidatos."""
    for p in candidates:
        if p.exists():
            return p
    return None


def find_sueiras_root() -> Optional[Path]:
    """Busca sueiras en rutas montadas y luego intenta descarga."""
    all_candidates = list(SUEIRAS_CANDIDATE_ROOTS)
    
    for item in INPUT_BASE.iterdir() if INPUT_BASE.exists() else []:
        name_lower = item.name.lower()
        if ('handwrit' in name_lower or 'sueiras' in name_lower) \
           and 'iam' not in name_lower and 'spanish' not in name_lower:
            all_candidates.append(item)
        if item.is_dir() and item.name == 'datasets':
            for owner in item.iterdir():
                if not owner.is_dir():
                    continue
                for ds in owner.iterdir():
                    name_lower = ds.name.lower()
                    if ('handwrit' in name_lower or 'sueiras' in name_lower) \
                       and 'iam' not in name_lower:
                        all_candidates.append(ds)
    
    for p in all_candidates:
        if p.exists() and any(p.rglob('*')):
            print(f'  sueiras encontrado en: {p}')
            return p
    
    for prev in [
        SUEIRAS_WORK_DIR / 'handwritting_characters_database',
        SUEIRAS_WORK_DIR / 'from_kaggle',
        SUEIRAS_WORK_DIR / 'from_github',
    ]:
        if prev.exists() and any(prev.rglob('*')):
            print(f'  sueiras encontrado (descarga previa): {prev}')
            return prev
    
    # Intentar descarga
    print('  Intentando descargar sueiras...')
    try:
        import kagglehub
        for slug in ['sueiras/handwritting-characters-database',
                     'sueiras/handwriting-characters-database']:
            try:
                path = kagglehub.dataset_download(slug)
                src = Path(path)
                if src.exists() and any(src.rglob('*')):
                    print(f'  ✓ sueiras descargado: {src}')
                    return src
            except Exception:
                continue
    except ImportError:
        pass
    
    return None


def extract_sueiras_if_needed(root: Path) -> Path:
    """Extrae tar.gz si es necesario."""
    sample_imgs = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp'):
        sample_imgs += list(root.rglob(ext))[:5]
        if sample_imgs:
            break
    
    if sample_imgs:
        return root
    
    for tar_path in root.rglob('*.tar.gz'):
        print(f'  sueiras: extrayendo {tar_path.name}...')
        SUEIRAS_EXTRACT_DIR.mkdir(exist_ok=True)
        done_flag = SUEIRAS_EXTRACT_DIR / '.done'
        if done_flag.exists():
            return SUEIRAS_EXTRACT_DIR
        try:
            with tarfile.open(tar_path, 'r:gz') as tar:
                tar.extractall(SUEIRAS_EXTRACT_DIR)
            done_flag.touch()
            return SUEIRAS_EXTRACT_DIR
        except Exception as e:
            print(f'  Error: {e}')
    
    return root


SUEIRAS_SLUG2CHAR = {
    **VERACK_SLUG2CHAR,
    'enye_min': 'ñ', 'enye_may': 'Ñ',
    'a_acute': 'á', 'e_acute': 'é', 'i_acute': 'í',
    'o_acute': 'ó', 'u_acute': 'ú',
    'A_acute': 'Á', 'E_acute': 'É', 'I_acute': 'Í',
    'O_acute': 'Ó', 'U_acute': 'Ú',
    'u_umlaut': 'ü', 'U_umlaut': 'Ü',
}


def sueiras_folder_to_char(folder_name: str) -> Optional[str]:
    idx = char_to_idx(folder_name)
    if idx is not None:
        return IDX2CHAR[idx]
    nfc = unicodedata.normalize('NFC', folder_name)
    idx = char_to_idx(nfc)
    if idx is not None:
        return IDX2CHAR[idx]
    for variant in [folder_name, folder_name.lower(), folder_name.upper()]:
        if variant in SUEIRAS_SLUG2CHAR:
            ch = SUEIRAS_SLUG2CHAR[variant]
            idx = char_to_idx(ch)
            if idx is not None:
                return ch
    if len(folder_name) == 1:
        idx = char_to_idx(folder_name)
        if idx is not None:
            return folder_name
    import re
    m = re.match(r'^cls(\d+)$', folder_name, re.I)
    if m:
        ci = int(m.group(1))
        if ci < len(emnist_chars):
            ch = emnist_chars[ci]
            idx = char_to_idx(ch)
            if idx is not None:
                return ch
        if ci in IDX2CHAR:
            return IDX2CHAR[ci]
    return None


def load_sueiras(root: Path) -> List[Dict]:
    records = []
    matched, skipped = 0, 0
    
    image_dirs = set()
    for sub in root.iterdir():
        if not sub.is_dir():
            continue
        has_img = any(sub.glob('*.png')) or any(sub.glob('*.jpg'))
        if has_img:
            image_dirs.add(sub)
        for sub2 in sub.iterdir():
            if not sub2.is_dir():
                continue
            has_img2 = any(sub2.glob('*.png')) or any(sub2.glob('*.jpg'))
            if has_img2:
                image_dirs.add(sub2)
    
    print(f'  Total carpetas con imágenes: {len(image_dirs)}')
    
    for folder in tqdm(sorted(image_dirs), desc='sueiras scan'):
        folder_name = folder.name
        if folder_name.startswith('.') or folder_name.lower() in (
            'images', 'labels', 'train', 'val', 'test', 'data',
            'curated', 'raw', 'processed'
        ):
            continue
        char = sueiras_folder_to_char(folder_name)
        if char is None:
            skipped += 1
            continue
        global_idx = char_to_idx(char)
        if global_idx is None:
            skipped += 1
            continue
        imgs = [f for f in folder.iterdir()
                if f.is_file() and f.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp')]
        for img_path in imgs:
            try:
                im = cv2.imread(str(img_path))
                if im is None:
                    continue
                h, w = im.shape[:2]
                if w > 4 * h:
                    continue
                records.append({
                    'image_path': str(img_path),
                    'label': char,
                    'class_idx': global_idx,
                    'source': 'sueiras',
                    'width': w,
                    'height': h,
                })
                matched += 1
            except Exception:
                continue
    
    print(f'  sueiras: {matched} imgs | {skipped} carpetas skip')
    return records


# ══════════════════════════════════════════════════════════════════════════════
# EJECUCIÓN: Skip si es el mismo dataset que verack
# ══════════════════════════════════════════════════════════════════════════════

print('=== Cargando sueiras handwritting_characters_database ===')

sueiras_records: List[Dict] = []

sueiras_root_candidate = find_existing_path(SUEIRAS_CANDIDATE_ROOTS)

skip_sueiras = False
if verack_root is not None and sueiras_root_candidate is not None:
    try:
        same_path = (verack_root.resolve() == sueiras_root_candidate.resolve())
    except Exception:
        same_path = (str(verack_root) == str(sueiras_root_candidate))
    
    if same_path:
        print(f'  ⚠️ sueiras apunta al mismo path que verack: {sueiras_root_candidate}')
        print(f'  → Saltando para evitar duplicados y ahorrar tiempo')
        skip_sueiras = True

if not skip_sueiras:
    sueiras_root = find_sueiras_root()
    if sueiras_root is not None:
        # Verificar de nuevo contra verack
        if verack_root is not None:
            try:
                same = (verack_root.resolve() == sueiras_root.resolve())
            except Exception:
                same = (str(verack_root) == str(sueiras_root))
            if same:
                print(f'  ⚠️ sueiras resolvió al mismo path que verack — saltando')
                skip_sueiras = True
        
        if not skip_sueiras:
            sueiras_root = extract_sueiras_if_needed(sueiras_root)
            sueiras_records = load_sueiras(sueiras_root)
    else:
        print('  sueiras no disponible')

print(f'sueiras records: {len(sueiras_records)}')

## Cell 7 — Phase A: Merge y deduplicación (FIX: manejo de DataFrame vacío)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Phase A: Merge, deduplicación + indexar EMNIST para Accent Aug
# ══════════════════════════════════════════════════════════════════════════════

SCHEMA_COLS = ['image_path', 'label', 'class_idx', 'source', 'width', 'height']

all_records = emnist_records + verack_records + sueiras_records

if len(all_records) == 0:
    print('WARNING: ningún dataset cargado — el DataFrame está vacío')
    df = pd.DataFrame(columns=SCHEMA_COLS)
    df = df.astype({'class_idx': int, 'width': int, 'height': int})
else:
    df = pd.DataFrame(all_records, columns=SCHEMA_COLS)
    df = df.drop_duplicates(subset=['image_path']).reset_index(drop=True)
    df = df[df['label'].notna()]
    df = df[df['class_idx'].between(0, NUM_CLASSES - 1)]
    df['class_idx'] = df['class_idx'].astype(int)

print(f'Registros reales totales: {len(df)}')

# ── Cobertura actual ─────────────────────────────────────────────────────────
covered = set(df['class_idx'].unique()) if len(df) > 0 else set()
missing = set(range(NUM_CLASSES)) - covered
missing_chars = [IDX2CHAR[i] for i in sorted(missing)]
print(f'Cobertura: {len(covered)}/107  |  Clases faltantes ({len(missing)}): {missing_chars}')

# ── NUEVO: Indexar paths de EMNIST por carácter ──────────────────────────────
# Esto es CRÍTICO para la Celda 8: necesitamos las imágenes manuscritas reales
# de 'a', 'e', 'i', 'o', 'u', 'n', etc. para generar 'á', 'é', 'í', etc.

EMNIST_PATHS_BY_CHAR: Dict[str, List[str]] = defaultdict(list)

if len(df) > 0:
    df_emnist = df[df['source'] == 'emnist']
    for _, row in df_emnist.iterrows():
        EMNIST_PATHS_BY_CHAR[row['label']].append(row['image_path'])

print(f'\n=== EMNIST paths indexados por carácter ===')
print(f'Caracteres con imágenes reales: {len(EMNIST_PATHS_BY_CHAR)}')

# Mostrar los caracteres base que usaremos para Accent Augmentation
print(f'\n=== Caracteres base disponibles para Accent Augmentation ===')
accent_aug_possible = {}
accent_aug_missing_base = []

for base_char, accented_list in ACCENT_MAP.items():
    n_base = len(EMNIST_PATHS_BY_CHAR.get(base_char, []))
    for acc_char, acc_type in accented_list:
        acc_idx = char_to_idx(acc_char)
        n_existing = len(df[df['class_idx'] == acc_idx]) if acc_idx is not None else 0
        if n_base > 0:
            accent_aug_possible[acc_char] = {
                'base_char': base_char,
                'accent_type': acc_type,
                'n_base_images': n_base,
                'n_existing': n_existing,
            }
            print(f'  {base_char} → {acc_char} ({acc_type}): '
                  f'{n_base} bases disponibles, {n_existing} existentes')
        else:
            accent_aug_missing_base.append((base_char, acc_char))
            print(f'  {base_char} → {acc_char} ({acc_type}): '
                  f'⚠️ SIN imágenes base reales')

if accent_aug_missing_base:
    print(f'\n⚠️ {len(accent_aug_missing_base)} pares sin base real: '
          f'{accent_aug_missing_base}')
else:
    print(f'\n✅ Todos los pares base→acentuada tienen imágenes EMNIST reales')

# ── Resumen de desbalance ────────────────────────────────────────────────────
print(f'\n=== Distribución por fuente ===')
if len(df) > 0:
    for src, grp in df.groupby('source'):
        n_imgs = len(grp)
        n_cls = grp['class_idx'].nunique()
        avg = n_imgs / max(n_cls, 1)
        print(f'  {src:20s}: {n_imgs:6d} imgs, {n_cls:3d} clases, '
              f'~{avg:.0f} imgs/clase')

print(f'\n=== Clases que necesitan datos (sin imágenes reales) ===')
for i in sorted(missing):
    ch = IDX2CHAR[i]
    in_accent_map = ch in accent_aug_possible
    tag = '← Accent Aug posible' if in_accent_map else '← Solo sintético'
    print(f'  [{i:3d}] {ch!r:5s} {tag}')

## Cell 8 — Phase A: Generación sintética + resize con padding (aspect-ratio preservado)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — Phase A: Accent Augmentation + Generación sintética mejorada
# ══════════════════════════════════════════════════════════════════════════════

# ────────────────────────────────────────────────────────────────────────────
# SECTION 1: Font discovery
# ────────────────────────────────────────────────────────────────────────────
FONT_SEARCH_PATHS = [
    '/usr/share/fonts', '/usr/local/share/fonts',
    '/usr/share/fonts/truetype', '/usr/share/fonts/truetype/dejavu',
    '/usr/share/fonts/truetype/liberation',
    '/usr/share/fonts/truetype/freefont',
    '/usr/share/fonts/truetype/noto',
]
found_fonts = []
for p in FONT_SEARCH_PATHS:
    found_fonts += glob.glob(f'{p}/**/*.ttf', recursive=True)
    found_fonts += glob.glob(f'{p}/**/*.otf', recursive=True)

usable_fonts = []
for fp in found_fonts[:50]:  # check more fonts
    try:
        ImageFont.truetype(fp, 60)
        usable_fonts.append(fp)
    except Exception:
        pass

print(f'Fuentes disponibles: {len(found_fonts)} encontradas, {len(usable_fonts)} usables')

# ────────────────────────────────────────────────────────────────────────────
# SECTION 2: Utility functions
# ────────────────────────────────────────────────────────────────────────────
def letterbox_resize(img_bgr: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """Resize con padding preservando aspect ratio."""
    h, w = img_bgr.shape[:2]
    scale = size / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 255, dtype=np.uint8)
    y0 = (size - new_h) // 2
    x0 = (size - new_w) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


# Indexar TODAS las imágenes reales por carácter (EMNIST + verack)
REAL_PATHS_BY_CHAR: Dict[str, List[str]] = defaultdict(list)
if len(df) > 0:
    for _, row in df.iterrows():
        REAL_PATHS_BY_CHAR[row['label']].append(row['image_path'])
print(f'Caracteres con imágenes reales: {len(REAL_PATHS_BY_CHAR)}')

# ════════════════════════════════════════════════════════════════════════════
# SECTION 3: ACCENT AUGMENTATION (NUEVO — CAMBIO CLAVE)
#
# Idea: tomar imagen manuscrita real de 'a' de EMNIST y dibujar un acento
# encima con cv2 → obtener una 'á' que se ve manuscrita y realista.
# Esto elimina el domain gap entre datos reales y sintéticos para las
# clases acentuadas (á, é, í, ó, ú, ü, ñ + mayúsculas).
# ════════════════════════════════════════════════════════════════════════════

def _find_char_bounds(gray: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    """Encuentra el bounding box del carácter (pixeles oscuros)."""
    mask = gray < 200
    if not mask.any():
        return None
    ys, xs = np.where(mask)
    return (int(xs.min()), int(ys.min()),
            int(xs.max() - xs.min() + 1), int(ys.max() - ys.min() + 1))


def _ensure_space_above(gray: np.ndarray, bounds: Tuple,
                        min_space: int = 20) -> Tuple[np.ndarray, Tuple]:
    """Si el carácter está muy arriba, lo baja para dejar espacio al acento."""
    x, y, w, h = bounds
    img_size = gray.shape[0]

    if y >= min_space:
        return gray, bounds

    shift = min_space - y + random.randint(3, 7)

    # Si desplazar empuja el carácter fuera de la imagen, escalar un poco
    if y + h + shift > img_size - 5:
        scale = 0.85
        center = img_size // 2
        M = cv2.getRotationMatrix2D((center, center), 0, scale)
        M[1, 2] += shift * 0.5
    else:
        M = np.float32([[1, 0, 0], [0, 1, shift]])

    gray = cv2.warpAffine(gray, M, (img_size, img_size), borderValue=255)

    # Recalcular bounds
    new_bounds = _find_char_bounds(gray)
    return gray, (new_bounds if new_bounds else bounds)


def _draw_acute(gray: np.ndarray, x: int, y: int, w: int, h: int) -> np.ndarray:
    """Dibuja acento agudo (´) encima del carácter."""
    cx = x + w // 2 + random.randint(-5, 5)
    top = y - random.randint(3, 8)

    length = random.randint(8, 15)
    angle_deg = random.uniform(50, 75)  # inclinación del acento
    dx = int(length * math.cos(math.radians(angle_deg)) / 2)
    dy = int(length * math.sin(math.radians(angle_deg)) / 2)

    pt1 = (cx - dx, top + dy)   # abajo-izquierda
    pt2 = (cx + dx, top - dy)   # arriba-derecha

    thickness = random.randint(1, 3)
    color = random.randint(0, 70)
    cv2.line(gray, pt1, pt2, color, thickness, cv2.LINE_AA)

    # A veces agregar un punto/engrosamiento en la punta (simula presión)
    if random.random() < 0.3:
        cv2.circle(gray, pt2, random.randint(1, 2), color, -1)

    return gray


def _draw_dieresis(gray: np.ndarray, x: int, y: int, w: int, h: int) -> np.ndarray:
    """Dibuja diéresis (¨) encima del carácter — dos puntos."""
    cx = x + w // 2
    top = y - random.randint(4, 10)
    
    sep = random.randint(5, 10)  # separación entre puntos
    radius = random.randint(1, 3)
    color = random.randint(0, 70)
    
    jx1 = random.randint(-2, 2)
    jy1 = random.randint(-2, 2)
    jx2 = random.randint(-2, 2)
    jy2 = random.randint(-2, 2)
    
    cv2.circle(gray, (cx - sep + jx1, top + jy1), radius, color, -1, cv2.LINE_AA)
    cv2.circle(gray, (cx + sep + jx2, top + jy2), radius, color, -1, cv2.LINE_AA)
    
    return gray


def _draw_tilde(gray: np.ndarray, x: int, y: int, w: int, h: int) -> np.ndarray:
    """Dibuja tilde (~) encima del carácter para ñ/Ñ."""
    cx = x + w // 2 + random.randint(-3, 3)
    top = y - random.randint(4, 10)
    
    # Tilde como curva sinusoidal con puntos
    half_w = random.randint(6, 12)
    amplitude = random.randint(2, 5)
    thickness = random.randint(1, 3)
    color = random.randint(0, 70)
    
    pts = []
    n_pts = 12
    for i in range(n_pts + 1):
        t = i / n_pts
        px = int(cx - half_w + 2 * half_w * t + random.randint(-1, 1))
        py = int(top + amplitude * math.sin(t * math.pi * 2) + random.randint(-1, 1))
        pts.append([px, py])
    
    pts = np.array(pts, dtype=np.int32)
    cv2.polylines(gray, [pts], False, color, thickness, cv2.LINE_AA)
    
    return gray


def add_accent_to_image(img_path: str, accent_type: str) -> Optional[np.ndarray]:
    """
    Toma una imagen manuscrita REAL y le dibuja un acento encima.
    
    Args:
        img_path: ruta a imagen EMNIST real (RGB, 128x128)
        accent_type: 'acute', 'dieresis', 'tilde'
    
    Returns:
        imagen RGB np.ndarray con acento dibujado, o None si falla
    """
    try:
        img = cv2.imread(img_path)
        if img is None:
            return None
        
        # Convertir a grayscale para trabajar
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Encontrar el bounding box del carácter
        bounds = _find_char_bounds(gray)
        if bounds is None:
            return None
        
        x, y, w, h = bounds
        
        # Si el carácter está muy arriba, desplazarlo para dejar espacio
        # Necesitamos ~20px arriba del carácter para el acento
        gray, bounds = _ensure_space_above(gray, bounds, min_space=18)
        x, y, w, h = bounds
        
        # Dibujar el acento según el tipo
        if accent_type == 'acute':
            gray = _draw_acute(gray, x, y, w, h)
        elif accent_type == 'dieresis':
            gray = _draw_dieresis(gray, x, y, w, h)
        elif accent_type == 'tilde':
            gray = _draw_tilde(gray, x, y, w, h)
        else:
            return None
        
        # ── Perturbaciones adicionales (para variedad) ──
        # Rotación leve
        if random.random() < 0.5:
            angle = random.uniform(-5, 5)
            center = (IMG_SIZE // 2, IMG_SIZE // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            gray = cv2.warpAffine(gray, M, (IMG_SIZE, IMG_SIZE), borderValue=255)
        
        # Erosión/dilatación (cambia grosor del trazo)
        if random.random() < 0.3:
            k = random.choice([2, 3])
            kernel = np.ones((k, k), np.uint8)
            if random.random() < 0.5:
                gray = cv2.erode(gray, kernel, iterations=1)
            else:
                gray = cv2.dilate(gray, kernel, iterations=1)
        
        # Ruido ligero
        if random.random() < 0.3:
            noise = np.random.randint(-15, 15, gray.shape, dtype=np.int16)
            gray = np.clip(gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        
        # Brillo/contraste leve
        if random.random() < 0.3:
            alpha = random.uniform(0.9, 1.1)
            beta = random.uniform(-10, 10)
            gray = np.clip(gray.astype(float) * alpha + beta, 0, 255).astype(np.uint8)
        
        # Convertir de vuelta a RGB
        rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        return rgb
    
    except Exception:
        return None


# ════════════════════════════════════════════════════════════════════════════
# SECTION 4: EJECUTAR ACCENT AUGMENTATION
# ════════════════════════════════════════════════════════════════════════════

print('=== Accent Augmentation: generando acentuadas desde EMNIST real ===')
accent_records: List[Dict] = []

for base_char, accented_list in tqdm(ACCENT_MAP.items(), desc='Accent Aug'):
    base_paths = REAL_PATHS_BY_CHAR.get(base_char, [])
    if not base_paths:
        print(f'  ⚠️ Sin imágenes base para {base_char!r} — saltando')
        continue
    
    for acc_char, acc_type in accented_list:
        acc_idx = char_to_idx(acc_char)
        if acc_idx is None:
            print(f'  ⚠️ {acc_char!r} no está en char_map — saltando')
            continue
        
        cls_dir = ACCENT_AUG_DIR / f'cls_{acc_idx:03d}_{acc_char}'
        cls_dir.mkdir(exist_ok=True)
        
        n_to_generate = ACCENT_SAMPLES_PER_BASE
        generated = 0
        attempts = 0
        max_attempts = n_to_generate * 3  # evitar loop infinito
        
        while generated < n_to_generate and attempts < max_attempts:
            # Elegir una imagen base aleatoria
            base_path = random.choice(base_paths)
            
            # Agregar acento
            result = add_accent_to_image(base_path, acc_type)
            if result is None:
                attempts += 1
                continue
            
            img_path = cls_dir / f'accent_{generated:05d}.png'
            cv2.imwrite(str(img_path), result)
            
            accent_records.append({
                'image_path': str(img_path),
                'label': acc_char,
                'class_idx': acc_idx,
                'source': 'accent_aug',   # fuente distinta para tracking
                'width': IMG_SIZE,
                'height': IMG_SIZE,
            })
            generated += 1
            attempts += 1
        
        print(f'  {base_char} → {acc_char} ({acc_type}): {generated} generadas')

print(f'\nTotal Accent Augmented: {len(accent_records)}')

# Verificar cobertura de acentuadas
accent_chars_generated = set(r['label'] for r in accent_records)
accent_chars_expected = set()
for _, acc_list in ACCENT_MAP.items():
    for acc_char, _ in acc_list:
        accent_chars_expected.add(acc_char)
accent_missing = accent_chars_expected - accent_chars_generated
if accent_missing:
    print(f'⚠️ Acentuadas sin generar: {accent_missing}')
else:
    print(f'✅ Todas las {len(accent_chars_expected)} clases acentuadas generadas')


# ════════════════════════════════════════════════════════════════════════════
# SECTION 5: Visualizar muestras de Accent Augmentation (control de calidad)
# ════════════════════════════════════════════════════════════════════════════

if accent_records:
    fig, axes = plt.subplots(3, 8, figsize=(20, 8))
    axes = axes.flatten()
    
    # Tomar muestras de diferentes clases acentuadas
    samples_by_char = defaultdict(list)
    for r in accent_records:
        samples_by_char[r['label']].append(r['image_path'])
    
    ax_i = 0
    for acc_char in sorted(samples_by_char.keys()):
        if ax_i >= len(axes):
            break
        paths = samples_by_char[acc_char]
        sample_path = random.choice(paths)
        img = cv2.imread(sample_path)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[ax_i].imshow(img_rgb)
            axes[ax_i].set_title(f'{acc_char!r} (accent_aug)', fontsize=9)
            axes[ax_i].axis('off')
            ax_i += 1
    
    # También mostrar el original para comparar
    for base_char in ['a', 'e', 'n', 'A']:
        if ax_i >= len(axes):
            break
        paths = REAL_PATHS_BY_CHAR.get(base_char, [])
        if paths:
            img = cv2.imread(random.choice(paths))
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[ax_i].imshow(img_rgb)
                axes[ax_i].set_title(f'{base_char!r} (EMNIST orig)', fontsize=9)
                axes[ax_i].axis('off')
                ax_i += 1
    
    for i in range(ax_i, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('Accent Augmentation: originales vs aumentadas', fontsize=14)
    plt.tight_layout()
    plt.savefig(WORK / 'accent_augmentation_samples.png', dpi=150)
    plt.close()
    print('Guardado accent_augmentation_samples.png')


# ════════════════════════════════════════════════════════════════════════════
# SECTION 6: Generación sintética (tipográfica) para clases restantes
# ════════════════════════════════════════════════════════════════════════════

def make_synthetic_image(char: str, font_path, img_size: int = IMG_SIZE) -> np.ndarray:
    """Renderiza un carácter con augmentaciones aleatorias."""
    img = Image.new('L', (img_size, img_size), color=255)
    draw = ImageDraw.Draw(img)
    font_size = random.randint(50, 85)
    try:
        font = ImageFont.truetype(font_path, font_size) \
               if isinstance(font_path, str) else ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), char, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (img_size - tw) // 2 - bbox[0] + random.randint(-8, 8)
    y = (img_size - th) // 2 - bbox[1] + random.randint(-8, 8)
    draw.text((x, y), char, fill=random.randint(0, 80), font=font)

    arr = np.array(img)

    # Rotación
    angle = random.uniform(-15, 15)
    M = cv2.getRotationMatrix2D((img_size // 2, img_size // 2), angle, 1.0)
    arr = cv2.warpAffine(arr, M, (img_size, img_size),
                          borderValue=255, flags=cv2.INTER_LINEAR)

    # Erosión / dilatación
    if random.random() < 0.5:
        k = random.choice([2, 3])
        kernel = np.ones((k, k), np.uint8)
        arr = cv2.erode(arr, kernel, 1) if random.random() < 0.5 \
              else cv2.dilate(arr, kernel, 1)

    # Ruido
    if random.random() < 0.4:
        noise = np.random.randint(-25, 25, arr.shape, dtype=np.int16)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Brillo / contraste
    if random.random() < 0.3:
        alpha = random.uniform(0.8, 1.2)
        beta  = random.uniform(-20, 20)
        arr = np.clip(arr.astype(float) * alpha + beta, 0, 255).astype(np.uint8)

    # NUEVO: Elastic-like warp para simular escritura manual
    if random.random() < 0.3:
        rows, cols = arr.shape
        dx = (np.random.rand(rows, cols).astype(np.float32) - 0.5) * 6
        dy = (np.random.rand(rows, cols).astype(np.float32) - 0.5) * 6
        x_map, y_map = np.meshgrid(np.arange(cols), np.arange(rows))
        map_x = (x_map + dx).astype(np.float32)
        map_y = (y_map + dy).astype(np.float32)
        arr = cv2.remap(arr, map_x, map_y, cv2.INTER_LINEAR, borderValue=255)

    arr_rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    return arr_rgb


# ════════════════════════════════════════════════════════════════════════════
# SECTION 7: Generación de trazos dibujados (líneas, curvas, círculos)
# ════════════════════════════════════════════════════════════════════════════

def _add_stroke_noise(arr: np.ndarray) -> np.ndarray:
    """Aplica ruido/variación a un trazo dibujado."""
    angle = random.uniform(-12, 12)
    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    arr = cv2.warpAffine(arr, M, (w, h), borderValue=255)

    if random.random() < 0.5:
        k = random.choice([2, 3])
        kernel = np.ones((k, k), np.uint8)
        if random.random() < 0.5:
            arr = cv2.erode(arr, kernel, 1)
        else:
            arr = cv2.dilate(arr, kernel, 1)

    if random.random() < 0.5:
        noise = np.random.randint(-20, 20, arr.shape, dtype=np.int16)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    if random.random() < 0.4:
        dx = random.randint(-8, 8)
        dy = random.randint(-8, 8)
        M_shift = np.float32([[1, 0, dx], [0, 1, dy]])
        arr = cv2.warpAffine(arr, M_shift, (w, h), borderValue=255)

    return arr


def make_stroke_image(stroke_type: str, img_size: int = IMG_SIZE) -> np.ndarray:
    """Genera una imagen de trazo básico dibujado con cv2."""
    canvas = np.full((img_size, img_size), 255, dtype=np.uint8)
    thickness = random.randint(2, 5)
    color = random.randint(0, 60)
    margin = random.randint(15, 30)
    center = img_size // 2
    jitter_x = random.randint(-10, 10)
    jitter_y = random.randint(-10, 10)

    if stroke_type == 'línea_vertical':
        x1 = center + jitter_x + random.randint(-3, 3)
        x2 = center + jitter_x + random.randint(-3, 3)
        y1 = margin + random.randint(-5, 5)
        y2 = img_size - margin + random.randint(-5, 5)
        pts = []
        n_pts = random.randint(5, 10)
        for i in range(n_pts + 1):
            t = i / n_pts
            px = int(x1 + (x2 - x1) * t + random.randint(-2, 2))
            py = int(y1 + (y2 - y1) * t)
            pts.append([px, py])
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    elif stroke_type == 'línea_horizontal':
        y1 = center + jitter_y + random.randint(-3, 3)
        y2 = center + jitter_y + random.randint(-3, 3)
        x1 = margin + random.randint(-5, 5)
        x2 = img_size - margin + random.randint(-5, 5)
        pts = []
        n_pts = random.randint(5, 10)
        for i in range(n_pts + 1):
            t = i / n_pts
            px = int(x1 + (x2 - x1) * t)
            py = int(y1 + (y2 - y1) * t + random.randint(-2, 2))
            pts.append([px, py])
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    elif stroke_type == 'línea_oblicua_derecha':
        x1 = margin + jitter_x
        y1 = margin + jitter_y
        x2 = img_size - margin + jitter_x
        y2 = img_size - margin + jitter_y
        pts = []
        n_pts = random.randint(5, 10)
        for i in range(n_pts + 1):
            t = i / n_pts
            px = int(x1 + (x2 - x1) * t + random.randint(-2, 2))
            py = int(y1 + (y2 - y1) * t + random.randint(-2, 2))
            pts.append([px, py])
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    elif stroke_type == 'línea_oblicua_izquierda':
        x1 = img_size - margin + jitter_x
        y1 = margin + jitter_y
        x2 = margin + jitter_x
        y2 = img_size - margin + jitter_y
        pts = []
        n_pts = random.randint(5, 10)
        for i in range(n_pts + 1):
            t = i / n_pts
            px = int(x1 + (x2 - x1) * t + random.randint(-2, 2))
            py = int(y1 + (y2 - y1) * t + random.randint(-2, 2))
            pts.append([px, py])
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    elif stroke_type == 'curva':
        curve_type = random.choice(['S', 'C', 'U', 'wave'])
        pts = []
        n_pts = 20
        if curve_type == 'S':
            for i in range(n_pts + 1):
                t = i / n_pts
                px = int(center + 35 * math.sin(t * math.pi * 2) + jitter_x)
                py = int(margin + (img_size - 2 * margin) * t + jitter_y)
                pts.append([px, py])
        elif curve_type == 'C':
            for i in range(n_pts + 1):
                t = i / n_pts
                angle_t = -math.pi / 2 + math.pi * t
                px = int(center + 35 * math.cos(angle_t) + jitter_x)
                py = int(center + 35 * math.sin(angle_t) + jitter_y)
                pts.append([px, py])
        elif curve_type == 'U':
            for i in range(n_pts + 1):
                t = i / n_pts
                angle_t = math.pi * t
                px = int(center + 35 * math.cos(angle_t) + jitter_x)
                py = int(center + 30 * math.sin(angle_t) + jitter_y)
                pts.append([px, py])
        else:
            for i in range(n_pts + 1):
                t = i / n_pts
                px = int(margin + (img_size - 2 * margin) * t)
                py = int(center + 25 * math.sin(t * math.pi * 3) + jitter_y)
                pts.append([px, py])
        for p in pts:
            p[0] += random.randint(-2, 2)
            p[1] += random.randint(-2, 2)
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    elif stroke_type == 'círculo':
        radius_x = random.randint(25, 45)
        radius_y = random.randint(25, 45)
        cx = center + jitter_x
        cy = center + jitter_y
        angle = random.uniform(-15, 15)
        pts = []
        n_pts = 30
        gap = random.uniform(0, 0.3)
        for i in range(n_pts + 1):
            t = i / n_pts * (2 * math.pi - gap)
            px = int(cx + radius_x * math.cos(t + math.radians(angle))
                     + random.randint(-2, 2))
            py = int(cy + radius_y * math.sin(t + math.radians(angle))
                     + random.randint(-2, 2))
            pts.append([px, py])
        pts = np.array(pts, dtype=np.int32)
        cv2.polylines(canvas, [pts], False, color, thickness, cv2.LINE_AA)

    canvas = _add_stroke_noise(canvas)
    canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
    return canvas_rgb


STROKE_CLASSES = {
    'línea_vertical':          101,
    'línea_horizontal':        102,
    'línea_oblicua_derecha':   103,
    'línea_oblicua_izquierda': 104,
    'curva':                   105,
    'círculo':                 106,
}

for stroke_name, expected_idx in STROKE_CLASSES.items():
    actual = IDX2CHAR.get(expected_idx, '???')
    assert actual == stroke_name, \
        f"Mismatch: IDX2CHAR[{expected_idx}]={actual!r} != {stroke_name!r}"
print('✅ Mapeo de trazos verificado')


# ════════════════════════════════════════════════════════════════════════════
# SECTION 8: Generación sintética principal
# ════════════════════════════════════════════════════════════════════════════

# Contar imágenes existentes incluyendo accent_aug
accent_counts = Counter(r['class_idx'] for r in accent_records)
class_counts = df.groupby('class_idx').size().to_dict() if len(df) > 0 else {}

# Sumar accent_aug al conteo
for ci, cnt in accent_counts.items():
    class_counts[ci] = class_counts.get(ci, 0) + cnt

synth_records: List[Dict] = []

print(f'\n=== Generación sintética (tipográfica + trazos) ===')
for idx in tqdm(range(NUM_CLASSES), desc='Generación sintética'):
    char = IDX2CHAR[idx]
    current = class_counts.get(idx, 0)
    is_stroke = char in STROKE_CLASSES

    if is_stroke:
        n_synth = max(SYNTH_PER_CLASS_STROKE - current, 0)
    elif current == 0:
        n_synth = SYNTH_PER_CLASS
    elif current < MIN_REAL_SAMPLES:
        n_synth = MIN_REAL_SAMPLES - current
    else:
        # NUEVO: incluso clases con muchos datos reales, agregar algunos
        # sintéticos para regularización (pero pocos)
        n_synth = 0

    if n_synth <= 0:
        continue

    cls_dir = SYNTH_DIR / f'cls_{idx:03d}'
    cls_dir.mkdir(exist_ok=True)

    for i in range(n_synth):
        try:
            if is_stroke:
                arr = make_stroke_image(char, IMG_SIZE)
            else:
                fp = random.choice(usable_fonts) if usable_fonts else None
                arr = make_synthetic_image(char, fp)

            img_path = cls_dir / f'synth_{i:05d}.png'
            cv2.imwrite(str(img_path), arr)
            synth_records.append({
                'image_path': str(img_path),
                'label': char,
                'class_idx': idx,
                'source': 'synthetic',
                'width': IMG_SIZE,
                'height': IMG_SIZE,
            })
        except Exception:
            continue

print(f'Sintéticos tipográficos/trazos: {len(synth_records)}')


# ════════════════════════════════════════════════════════════════════════════
# SECTION 9: Hard pair synthesis (size-aware para case confusion)
# ════════════════════════════════════════════════════════════════════════════

def make_size_aware_synthetic(char: str, font_path, img_size: int = IMG_SIZE) -> np.ndarray:
    """Genera imágenes con tamaño diferenciado para may/min."""
    is_upper = char.isupper() and char.isalpha()
    is_lower = char.islower() and char.isalpha()
    is_digit = char.isdigit()

    if is_upper:
        font_size = random.randint(70, 95)
        y_offset = random.randint(-5, 5)
    elif is_lower:
        font_size = random.randint(40, 65)
        y_offset = random.randint(5, 20)
    elif is_digit:
        font_size = random.randint(55, 75)
        y_offset = random.randint(-5, 5)
    else:
        font_size = random.randint(50, 85)
        y_offset = 0

    img = Image.new('L', (img_size, img_size), color=255)
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype(font_path, font_size) \
               if isinstance(font_path, str) else ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), char, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (img_size - tw) // 2 - bbox[0] + random.randint(-5, 5)
    y = (img_size - th) // 2 - bbox[1] + y_offset
    draw.text((x, y), char, fill=random.randint(0, 80), font=font)

    arr = np.array(img)

    angle = random.uniform(-12, 12)
    M = cv2.getRotationMatrix2D((img_size // 2, img_size // 2), angle, 1.0)
    arr = cv2.warpAffine(arr, M, (img_size, img_size), borderValue=255)

    if random.random() < 0.5:
        k = random.choice([2, 3])
        kernel = np.ones((k, k), np.uint8)
        arr = cv2.erode(arr, kernel, 1) if random.random() < 0.5 \
              else cv2.dilate(arr, kernel, 1)

    if random.random() < 0.4:
        noise = np.random.randint(-20, 20, arr.shape, dtype=np.int16)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # NUEVO: elastic warp
    if random.random() < 0.3:
        rows, cols = arr.shape
        dx = (np.random.rand(rows, cols).astype(np.float32) - 0.5) * 5
        dy = (np.random.rand(rows, cols).astype(np.float32) - 0.5) * 5
        x_map, y_map = np.meshgrid(np.arange(cols), np.arange(rows))
        map_x = (x_map + dx).astype(np.float32)
        map_y = (y_map + dy).astype(np.float32)
        arr = cv2.remap(arr, map_x, map_y, cv2.INTER_LINEAR, borderValue=255)

    return cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)


confused_extra_records: List[Dict] = []

for char, _ in tqdm(CONFUSED_PAIRS, desc='Hard pair synthesis'):
    global_idx = char_to_idx(char)
    if global_idx is None:
        continue

    cls_dir = SYNTH_DIR / f'cls_{global_idx:03d}_hard'
    cls_dir.mkdir(exist_ok=True)

    for i in range(EXTRA_PER_CONFUSED):
        fp = random.choice(usable_fonts) if usable_fonts else None
        try:
            arr = make_size_aware_synthetic(char, fp)
            img_path = cls_dir / f'hard_{i:05d}.png'
            cv2.imwrite(str(img_path), arr)
            confused_extra_records.append({
                'image_path': str(img_path),
                'label': char,
                'class_idx': global_idx,
                'source': 'synthetic_hard',
                'width': IMG_SIZE,
                'height': IMG_SIZE,
            })
        except Exception:
            continue

print(f'Hard pair extras: {len(confused_extra_records)}')

synth_records += confused_extra_records


# ════════════════════════════════════════════════════════════════════════════
# SECTION 10: FONT SUPPLEMENT para TODAS las clases (NUEVO — CRÍTICO)
#
# Problema: las clases EMNIST (letras, dígitos) solo tienen datos manuscritos.
# El test real y muchos usos generan imágenes con fonts tipográficas.
# Sin este suplemento, el modelo NO reconoce letras/dígitos en fonts.
#
# Solución: agregar ~200 imágenes font-rendered por clase, incluso para
# clases que ya tienen datos EMNIST abundantes.
# ════════════════════════════════════════════════════════════════════════════

print(f'\n=== Font Supplement: imágenes tipográficas para TODAS las clases ===')
font_supplement_records: List[Dict] = []

# Clases que YA son solo sintéticas no necesitan más fonts
# (ya tienen SYNTH_PER_CLASS = 500 imágenes font-rendered)
classes_already_synthetic = set()
for idx in range(NUM_CLASSES):
    char = IDX2CHAR[idx]
    is_stroke = char in STROKE_CLASSES
    has_real = len(REAL_PATHS_BY_CHAR.get(char, [])) > 0
    has_accent = any(r['class_idx'] == idx for r in accent_records)
    if not has_real and not has_accent and not is_stroke:
        classes_already_synthetic.add(idx)

for idx in tqdm(range(NUM_CLASSES), desc='Font supplement'):
    char = IDX2CHAR[idx]
    
    # Skip trazos (no son caracteres tipográficos)
    if char in STROKE_CLASSES:
        continue
    
    # Skip clases que ya son 100% sintéticas (ya tienen fonts suficientes)
    if idx in classes_already_synthetic:
        continue
    
    cls_dir = SYNTH_DIR / f'cls_{idx:03d}_font'
    cls_dir.mkdir(exist_ok=True)
    
    for i in range(FONT_SUPPLEMENT_PER_CLASS):
        fp = random.choice(usable_fonts) if usable_fonts else None
        try:
            arr = make_synthetic_image(char, fp)
            img_path = cls_dir / f'font_{i:05d}.png'
            cv2.imwrite(str(img_path), arr)
            font_supplement_records.append({
                'image_path': str(img_path),
                'label': char,
                'class_idx': idx,
                'source': 'synthetic_font',
                'width': IMG_SIZE,
                'height': IMG_SIZE,
            })
        except Exception:
            continue

synth_records += font_supplement_records

# Resumen
font_classes = set(r['class_idx'] for r in font_supplement_records)
print(f'Font supplement: {len(font_supplement_records)} imágenes '
      f'para {len(font_classes)} clases')
print(f'  → Ahora letras y dígitos TAMBIÉN tienen datos font-rendered')
print(f'  → El modelo podrá reconocer tanto manuscrito como tipográfico')

print(f'\n{"="*60}')
print(f'=== RESUMEN DE GENERACIÓN DE DATOS ===')
print(f'{"="*60}')
print(f'Accent Augmented (desde EMNIST real): {len(accent_records):>6}')
print(f'Sintéticos (tipográficos + trazos):   {len(synth_records):>6}')
print(f'  - Base tipográficos:                {len(synth_records) - len(confused_extra_records):>6}')
print(f'  - Hard pair extras:                 {len(confused_extra_records):>6}')
print(f'EMNIST real:                          {len(emnist_records):>6}')
print(f'Verack real:                          {len(verack_records):>6}')
print(f'{"="*60}')
total = len(emnist_records) + len(verack_records) + len(accent_records) + len(synth_records)
print(f'TOTAL estimado:                       {total:>6}')

## Cell 9 — Phase A: Split estratificado y guardado de metadata

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — Phase A: Split estratificado + test honesto
# ══════════════════════════════════════════════════════════════════════════════

# ── Merge de TODAS las fuentes ───────────────────────────────────────────────
df_synth  = pd.DataFrame(synth_records, columns=SCHEMA_COLS) if synth_records \
            else pd.DataFrame(columns=SCHEMA_COLS)
df_accent = pd.DataFrame(accent_records, columns=SCHEMA_COLS) if accent_records \
            else pd.DataFrame(columns=SCHEMA_COLS)

df_all = pd.concat([df, df_accent, df_synth], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['image_path']).reset_index(drop=True)
df_all['class_idx'] = df_all['class_idx'].astype(int)

# ── Verificar cobertura ──────────────────────────────────────────────────────
covered_final = set(df_all['class_idx'].unique())
still_missing = set(range(NUM_CLASSES)) - covered_final
if still_missing:
    print(f'⚠️ Clases sin datos: {[IDX2CHAR[i] for i in sorted(still_missing)]}')
else:
    print('✅ Todas las 107 clases cubiertas')

# ── Distribución por fuente ──────────────────────────────────────────────────
print(f'\n=== Distribución por fuente ===')
for src, grp in df_all.groupby('source'):
    n_imgs = len(grp)
    n_cls = grp['class_idx'].nunique()
    avg = n_imgs / max(n_cls, 1)
    print(f'  {src:20s}: {n_imgs:6d} imgs, {n_cls:3d} clases, ~{avg:.0f} imgs/clase')
print(f'  {"TOTAL":20s}: {len(df_all):6d} imgs')

# ══════════════════════════════════════════════════════════════════════════════
# SPLIT ESTRATEGIA: 
# Para clases con datos REALES (emnist, verack, accent_aug):
#   → Asegurar que TEST y VAL contengan datos reales/accent_aug
#   → Los sintéticos van preferentemente a TRAIN
# Para clases 100% sintéticas:
#   → Split normal estratificado
# ══════════════════════════════════════════════════════════════════════════════

# Identificar qué clases tienen datos "realistas" (reales + accent_aug)
REAL_SOURCES = {'emnist', 'verack', 'accent_aug'}

df_all['is_real'] = df_all['source'].isin(REAL_SOURCES)

classes_with_real = set(
    df_all[df_all['is_real']].groupby('class_idx')
    .filter(lambda g: len(g) >= 10)['class_idx'].unique()
)
classes_synth_only = set(range(NUM_CLASSES)) - classes_with_real

print(f'\n=== Estrategia de split ===')
print(f'Clases con datos realistas (≥10 imgs): {len(classes_with_real)}')
print(f'Clases solo sintéticas:                {len(classes_synth_only)}')
print(f'Clases solo sintéticas: {[IDX2CHAR[i] for i in sorted(classes_synth_only)]}')

# ── Split separado para cada grupo ──────────────────────────────────────────
df_all['split'] = ''

# GRUPO 1: Clases con datos realistas
# Para estas: test y val SOLO con datos realistas, train con todo
mask_real_classes = df_all['class_idx'].isin(classes_with_real)
df_real_cls = df_all[mask_real_classes].copy()

# Separar datos realistas de sintéticos dentro de estas clases
df_real_imgs = df_real_cls[df_real_cls['is_real']].copy()
df_synth_imgs = df_real_cls[~df_real_cls['is_real']].copy()

print(f'\nClases con datos realistas:')
print(f'  Imágenes realistas: {len(df_real_imgs)}')
print(f'  Imágenes sintéticas (complemento): {len(df_synth_imgs)}')

# Split de imágenes realistas: 70% train, 15% val, 15% test
labels_real = df_real_imgs['class_idx'].values
idx_real = np.arange(len(df_real_imgs))

idx_real_train, idx_real_temp = train_test_split(
    idx_real, test_size=0.30, stratify=labels_real, random_state=SEED
)
idx_real_val, idx_real_test = train_test_split(
    idx_real_temp, test_size=0.50,
    stratify=labels_real[idx_real_temp], random_state=SEED
)

# Asignar splits a las imágenes realistas
real_indices = df_real_imgs.index.values
df_all.loc[real_indices[idx_real_train], 'split'] = 'train'
df_all.loc[real_indices[idx_real_val],   'split'] = 'val'
df_all.loc[real_indices[idx_real_test],  'split'] = 'test'

# Sintéticos de estas clases → TODOS a train
synth_indices = df_synth_imgs.index.values
df_all.loc[synth_indices, 'split'] = 'train'

# GRUPO 2: Clases solo sintéticas
# Split normal 80/10/10
mask_synth_classes = df_all['class_idx'].isin(classes_synth_only)
df_synth_cls = df_all[mask_synth_classes].copy()

if len(df_synth_cls) > 0:
    labels_synth = df_synth_cls['class_idx'].values
    idx_synth_all = np.arange(len(df_synth_cls))
    
    idx_s_train, idx_s_temp = train_test_split(
        idx_synth_all, test_size=0.20, stratify=labels_synth, random_state=SEED
    )
    idx_s_val, idx_s_test = train_test_split(
        idx_s_temp, test_size=0.50,
        stratify=labels_synth[idx_s_temp], random_state=SEED
    )
    
    synth_cls_indices = df_synth_cls.index.values
    df_all.loc[synth_cls_indices[idx_s_train], 'split'] = 'train'
    df_all.loc[synth_cls_indices[idx_s_val],   'split'] = 'val'
    df_all.loc[synth_cls_indices[idx_s_test],  'split'] = 'test'

# ── Verificar que no queden filas sin split ──────────────────────────────────
no_split = df_all[df_all['split'] == '']
if len(no_split) > 0:
    print(f'⚠️ {len(no_split)} filas sin split asignado — asignando a train')
    df_all.loc[df_all['split'] == '', 'split'] = 'train'

# ── Eliminar columna auxiliar ────────────────────────────────────────────────
df_all = df_all.drop(columns=['is_real'])

# ── Verificar que TODAS las clases tienen datos en val y test ────────────────
for split_name in ['val', 'test']:
    split_classes = set(df_all[df_all['split'] == split_name]['class_idx'].unique())
    missing_in_split = set(range(NUM_CLASSES)) - split_classes
    if missing_in_split:
        print(f'⚠️ {split_name} le faltan {len(missing_in_split)} clases: '
              f'{[IDX2CHAR[i] for i in sorted(missing_in_split)][:10]}...')
    else:
        print(f'✅ {split_name}: todas las 107 clases presentes')

# ── Guardar ──────────────────────────────────────────────────────────────────
df_all = df_all[['image_path', 'label', 'class_idx', 'source', 'split', 'width', 'height']]
df_all.to_parquet(META_PATH, index=False)

print(f'\n=== Guardado merged_metadata.parquet | {len(df_all)} filas ===')
print(df_all['split'].value_counts())

# ── Diagnóstico detallado por split ──────────────────────────────────────────
print(f'\n=== Composición de cada split por fuente ===')
for split_name in ['train', 'val', 'test']:
    df_split = df_all[df_all['split'] == split_name]
    print(f'\n{split_name.upper()} ({len(df_split)} imgs):')
    for src, grp in df_split.groupby('source'):
        print(f'  {src:20s}: {len(grp):6d} imgs, {grp["class_idx"].nunique():3d} clases')

# ── Diagnóstico: test set composición para clases acentuadas ─────────────────
print(f'\n=== Test set: clases acentuadas (verificar datos realistas) ===')
accent_chars_check = ['á', 'é', 'í', 'ó', 'ú', 'ü', 'ñ', 'Á', 'É', 'Í', 'Ó', 'Ú', 'Ñ']
df_test_check = df_all[df_all['split'] == 'test']
for ch in accent_chars_check:
    idx = char_to_idx(ch)
    if idx is None:
        continue
    ch_test = df_test_check[df_test_check['class_idx'] == idx]
    sources = ch_test['source'].value_counts().to_dict()
    total = len(ch_test)
    tag = '✅' if 'accent_aug' in sources or 'emnist' in sources or 'verack' in sources else '⚠️ solo sintético'
    print(f'  {ch!r:4s} (idx={idx:3d}): {total:4d} test imgs | fuentes: {sources} {tag}')

# ── Coverage report ──────────────────────────────────────────────────────────
per_class_counts = df_all.groupby('class_idx').size().to_dict()
per_source_cov = {src: int(g['class_idx'].nunique())
                  for src, g in df_all.groupby('source')}
coverage_report = {
    'total_classes': NUM_CLASSES,
    'covered_classes': len(covered_final),
    'missing_classes': [IDX2CHAR[i] for i in sorted(still_missing)],
    'classes_with_real_data': len(classes_with_real),
    'classes_synth_only': len(classes_synth_only),
    'synth_only_chars': [IDX2CHAR[i] for i in sorted(classes_synth_only)],
    'per_source_coverage': per_source_cov,
    'per_class_sample_count': {IDX2CHAR[k]: int(v) for k, v in per_class_counts.items()},
    'split_strategy': 'real_data_in_test_for_real_classes',
}
with open(COVERAGE_PATH, 'w') as f:
    json.dump(coverage_report, f, indent=2, ensure_ascii=False)
print('\nGuardado class_coverage_report.json')

## Cell 10 — Phase B: Dataset con letterbox resize + Augmentaciones mejoradas

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10 — Phase B: Dataset + Augmentaciones optimizadas para handwriting
# ══════════════════════════════════════════════════════════════════════════════

# ── Augmentaciones de TRAIN ──────────────────────────────────────────────────
# Diseñadas para simular variaciones reales de escritura manuscrita:
# - ElasticTransform fuerte: simula deformación del papel/mano
# - Erosión/dilatación implícita via morphological transforms
# - Rotación/escala moderada (la gente no escribe a 45°)
# - Ruido y blur suaves (papel, escáner)

train_transform = A.Compose([
    # ── Geométricas (simular variación de escritura) ──
    A.ShiftScaleRotate(
        shift_limit=0.08, scale_limit=0.15, rotate_limit=12,
        border_mode=cv2.BORDER_CONSTANT, value=255, p=0.7
    ),
    A.ElasticTransform(
        alpha=1.5, sigma=25, p=0.4   # MÁS fuerte: simula trazo manual
    ),
    A.Perspective(scale=(0.02, 0.06), p=0.25),
    A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),

    # ── Simular variaciones de grosor de trazo ──
    A.OneOf([
        A.Morphological(scale=(2, 3), operation='erosion', p=1.0),   # trazo más fino
        A.Morphological(scale=(2, 3), operation='dilation', p=1.0),  # trazo más grueso
    ], p=0.3),

    # ── Ruido / blur (simular papel, escáner, cámara) ──
    A.OneOf([
        A.GaussNoise(var_limit=(5, 30), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.15), p=1.0),
    ], p=0.3),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=3, p=1.0),
    ], p=0.2),

    # ── Brillo / contraste (simular diferentes condiciones de iluminación) ──
    A.RandomBrightnessContrast(
        brightness_limit=0.25, contrast_limit=0.25, p=0.4
    ),
    A.RandomGamma(gamma_limit=(80, 120), p=0.2),

    # ── Simular degradación de imagen ──
    A.ImageCompression(quality_lower=75, quality_upper=100, p=0.1),

    # ── Dropout (simular tinta faltante, oclusiones parciales) ──
    A.CoarseDropout(
        max_holes=3, max_height=12, max_width=12,
        fill_value=255, p=0.15  # Menos agresivo que antes
    ),

    # ── Normalización ──
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# ── TTA transforms (5 variaciones) ──────────────────────────────────────────
tta_transforms = [
    # TTA 0: sin augmentación (baseline)
    val_transform,

    # TTA 1: rotación + escala leve
    A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.08, rotate_limit=7,
            border_mode=cv2.BORDER_CONSTANT, value=255, p=1.0
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]),

    # TTA 2: blur suave
    A.Compose([
        A.GaussianBlur(blur_limit=(3, 3), p=1.0),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]),

    # TTA 3: brillo/contraste
    A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15, p=1.0
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]),

    # TTA 4: elastic leve (simula variación de trazo)
    A.Compose([
        A.ElasticTransform(alpha=0.5, sigma=20, p=1.0),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]),
]


# ══════════════════════════════════════════════════════════════════════════════
# Dataset
# ══════════════════════════════════════════════════════════════════════════════

class CharDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform,
                 img_size: int = IMG_SIZE, use_letterbox: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size
        self.use_letterbox = use_letterbox

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = int(row['class_idx'])

        img = cv2.imread(str(row['image_path']))
        if img is None:
            img = np.full((self.img_size, self.img_size, 3), 255, dtype=np.uint8)
        else:
            if self.use_letterbox:
                img = letterbox_resize(img, self.img_size)
            else:
                img = cv2.resize(img, (self.img_size, self.img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tensor = self.transform(image=img)['image']
        return tensor, label


# ══════════════════════════════════════════════════════════════════════════════
# Cargar metadata y crear splits
# ══════════════════════════════════════════════════════════════════════════════

meta = pd.read_parquet(META_PATH)
df_train = meta[meta['split'] == 'train'].copy()
df_val   = meta[meta['split'] == 'val'].copy()
df_test  = meta[meta['split'] == 'test'].copy()

print(f'Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}')

# ── Diagnóstico del train set ────────────────────────────────────────────────
print(f'\nTrain set composición:')
for src, grp in df_train.groupby('source'):
    print(f'  {src:20s}: {len(grp):6d} imgs, {grp["class_idx"].nunique():3d} clases')

# ══════════════════════════════════════════════════════════════════════════════
# WeightedRandomSampler MEJORADO
#
# Estrategia de pesos:
# 1. Base: inverse frequency (clases con menos datos → más peso)
# 2. Boost: accent_aug y verack reciben peso extra (son más valiosas
#    que EMNIST porque cubren clases difíciles)
# 3. Penalty: synthetic puro recibe peso reducido (evitar que domine)
# ══════════════════════════════════════════════════════════════════════════════

train_labels = df_train['class_idx'].values
train_sources = df_train['source'].values

# Peso base por frecuencia de clase (inverse frequency)
class_counts_arr = np.bincount(train_labels, minlength=NUM_CLASSES).clip(min=1)
class_weights = 1.0 / class_counts_arr

# Source multipliers: ponderar más los datos valiosos
SOURCE_WEIGHT_MULTIPLIER = {
    'emnist':         1.0,   # datos reales estándar
    'verack':         1.5,   # datos reales españoles (más valiosos, son pocos)
    'accent_aug':     1.3,   # accent augmented (críticos para acentuadas)
    'synthetic':      0.7,   # sintéticos puros (menos peso, evitar domain overfit)
    'synthetic_hard': 0.9,# hard pairs (útiles pero sintéticos)
    'synthetic_font':  0.8,
}

sample_weights = np.zeros(len(df_train), dtype=np.float64)
for i in range(len(df_train)):
    lbl = train_labels[i]
    src = train_sources[i]
    base_w = class_weights[lbl]
    src_mult = SOURCE_WEIGHT_MULTIPLIER.get(src, 1.0)
    sample_weights[i] = base_w * src_mult

# Normalizar para que sumen a len(train)
sample_weights = sample_weights / sample_weights.sum() * len(df_train)

sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.float)
sampler = WeightedRandomSampler(
    sample_weights_tensor,
    num_samples=len(sample_weights_tensor),
    replacement=True
)

# ── Verificar distribución efectiva del sampler ──────────────────────────────
print(f'\n=== Verificación de WeightedRandomSampler ===')

# Simular 1 época de sampling
simulated_indices = list(sampler)[:len(df_train)]
simulated_labels = train_labels[simulated_indices]
simulated_sources = train_sources[simulated_indices]

sim_class_counts = np.bincount(simulated_labels, minlength=NUM_CLASSES)
print(f'Muestras por clase (simulación 1 época):')
print(f'  Min: {sim_class_counts.min()}  Max: {sim_class_counts.max()}  '
      f'Mean: {sim_class_counts.mean():.0f}  Std: {sim_class_counts.std():.0f}')

sim_source_counts = Counter(simulated_sources)
print(f'Muestras por fuente (simulación 1 época):')
for src, cnt in sorted(sim_source_counts.items()):
    pct = 100 * cnt / len(simulated_indices)
    print(f'  {src:20s}: {cnt:6d} ({pct:.1f}%)')


# ══════════════════════════════════════════════════════════════════════════════
# DataLoaders
# ══════════════════════════════════════════════════════════════════════════════

train_ds = CharDataset(df_train, train_transform, use_letterbox=True)
val_ds   = CharDataset(df_val,   val_transform,   use_letterbox=True)
test_ds  = CharDataset(df_test,  val_transform,   use_letterbox=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True  # drop_last para estabilidad
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'\n=== DataLoaders creados ===')
print(f'Train: {len(train_ds)} imgs, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} imgs, {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} imgs, {len(test_loader)} batches')

# ── Verificar un batch ───────────────────────────────────────────────────────
sample_batch, sample_labels = next(iter(train_loader))
print(f'\nSample batch shape: {sample_batch.shape}')
print(f'Sample labels: {sample_labels[:10].tolist()}')
print(f'Unique labels in batch: {len(sample_labels.unique())}')

## Cell 11 — Phase B: ArcFaceHead y OCRClassifier

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11 — ArcFaceHead + Projection Head + OCRClassifier + FocalLoss
# ══════════════════════════════════════════════════════════════════════════════

# ── Embedding dimension para projection head ─────────────────────────────────
EMBED_DIM = 512  # 1280 (backbone) → 512 (projection) → ArcFace


class FocalLoss(nn.Module):
    """
    Focal Loss: reduce el peso de ejemplos fáciles,
    enfoca el entrenamiento en los difíciles.
    
    Mejora: soporta class_weights opcionales para penalizar más
    clases específicas (ej: acentuadas).
    """
    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.05,
                 class_weights: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights  # shape: (NUM_CLASSES,)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(
            logits, targets,
            reduction='none',
            label_smoothing=self.label_smoothing,
            weight=self.class_weights  # per-class weighting
        )
        pt = torch.exp(-ce)  # probabilidad de la clase correcta
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


class ArcFaceHead(nn.Module):
    """
    ArcFace: Angular margin loss para embeddings discriminativos.
    Fuerza al modelo a separar clases similares (a/á, c/C, etc.)
    en el espacio angular.
    """
    def __init__(self, in_features: int, num_classes: int,
                 s: float = 30.0, m: float = 0.15):
        super().__init__()
        self.s = s
        self.m = m
        self.in_features = in_features
        self.num_classes = num_classes
        self.W = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.W)
        self.cos_m     = math.cos(m)
        self.sin_m     = math.sin(m)
        self.threshold = math.cos(math.pi - m)
        self.mm        = math.sin(math.pi - m) * m

    def forward(self, features, labels):
        x = F.normalize(features, dim=1)
        W = F.normalize(self.W, dim=1)
        cosine = F.linear(x, W)
        sine   = torch.sqrt((1.0 - cosine.pow(2)).clamp(min=1e-6))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.threshold, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return logits


class OCRClassifier(nn.Module):
    """
    Arquitectura:
      backbone (EfficientNetV2-S, 1280-dim)
        → projection head (1280 → 512, con BN + ReLU + Dropout)
          → ArcFace head (512 → NUM_CLASSES)
    
    El projection head es NUEVO y crítico:
    - Reduce 1280→512: más eficiente para cosine similarity en ArcFace
    - BatchNorm: estabiliza la distribución de embeddings
    - ReLU + Dropout: no-linealidad y regularización
    - Permite al modelo aprender features discriminativas específicas
      para pares confusos (a/á, c/C) en un espacio de menor dimensión
    """
    def __init__(self, num_classes: int = NUM_CLASSES,
                 backbone_name: str = 'tf_efficientnetv2_s',
                 embed_dim: int = EMBED_DIM):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0
        )
        feat_dim = self.backbone.num_features  # 1280 para efficientnetv2_s
        self.embed_dim = embed_dim

        # ── Projection Head (NUEVO) ──────────────────────────────────────
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT_RATE),
        )

        # ── ArcFace Head ─────────────────────────────────────────────────
        self.arcface = ArcFaceHead(embed_dim, num_classes, s=ARCFACE_S, m=ARCFACE_M)

    def get_embeddings(self, images: torch.Tensor) -> torch.Tensor:
        """Extrae embeddings 512-dim (útil para visualización/debug)."""
        features = self.backbone(images)
        return self.projector(features)

    def forward(self, images, labels=None):
        features = self.backbone(images)
        embeddings = self.projector(features)

        if labels is not None:
            # Training: ArcFace con margin
            return self.arcface(embeddings, labels)

        # Inference: cosine similarity sin margin
        x = F.normalize(embeddings, dim=1)
        W = F.normalize(self.arcface.W, dim=1)
        return F.linear(x, W) * self.arcface.s


# ══════════════════════════════════════════════════════════════════════════════
# Crear modelo
# ══════════════════════════════════════════════════════════════════════════════

model = OCRClassifier(num_classes=NUM_CLASSES).to(DEVICE)

# ── Verificar arquitectura ───────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_params = sum(p.numel() for p in model.backbone.parameters())
projector_params = sum(p.numel() for p in model.projector.parameters())
arcface_params = sum(p.numel() for p in model.arcface.parameters())

print(f'=== OCRClassifier v5 ===')
print(f'Backbone:   {backbone_params/1e6:.1f}M params '
      f'(feat_dim={model.backbone.num_features})')
print(f'Projector:  {projector_params/1e6:.2f}M params '
      f'({model.backbone.num_features}→{EMBED_DIM})')
print(f'ArcFace:    {arcface_params/1e6:.2f}M params '
      f'({EMBED_DIM}→{NUM_CLASSES})')
print(f'Total:      {total_params/1e6:.1f}M params '
      f'({trainable_params/1e6:.1f}M trainable)')

# ── Quick forward test ───────────────────────────────────────────────────────
with torch.no_grad():
    dummy_img = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    dummy_lbl = torch.tensor([0, 1]).to(DEVICE)

    # Test training mode
    train_out = model(dummy_img, dummy_lbl)
    assert train_out.shape == (2, NUM_CLASSES), \
        f'Training output shape mismatch: {train_out.shape}'

    # Test inference mode
    model.eval()
    inf_out = model(dummy_img)
    assert inf_out.shape == (2, NUM_CLASSES), \
        f'Inference output shape mismatch: {inf_out.shape}'

    # Test embeddings
    emb = model.get_embeddings(dummy_img)
    assert emb.shape == (2, EMBED_DIM), \
        f'Embedding shape mismatch: {emb.shape}'

    # ✅ FIX: Verificar que la escala s se aplica correctamente
    #    comparando output con coseno crudo × s
    x = F.normalize(emb, dim=1)
    W = F.normalize(model.arcface.W, dim=1)
    raw_cosine = F.linear(x, W)
    expected = raw_cosine * model.arcface.s
    assert torch.allclose(inf_out, expected, atol=1e-4), \
        f'ArcFace scale not applied correctly: ' \
        f'max_diff={(inf_out - expected).abs().max():.6f}'

    model.train()

print(f'\n✅ Forward test OK')
print(f'  Training output: {train_out.shape}, '
      f'range=[{train_out.min():.1f}, {train_out.max():.1f}]')
print(f'  Inference output: {inf_out.shape}, '
      f'range=[{inf_out.min():.1f}, {inf_out.max():.1f}]')
print(f'  Embeddings: {emb.shape}, '
      f'norm=[{emb.norm(dim=1).min():.2f}, {emb.norm(dim=1).max():.2f}]')
print(f'  Raw cosine range: [{raw_cosine.min():.3f}, {raw_cosine.max():.3f}]')
print(f'  After scale (×{model.arcface.s}): '
      f'[{inf_out.min():.1f}, {inf_out.max():.1f}]')

## Cell 12 — Phase B: Configuración de entrenamiento

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12 — Phase B: Configuración de entrenamiento optimizada v5
# ══════════════════════════════════════════════════════════════════════════════

# ── Calcular class weights para FocalLoss ────────────────────────────────────
# Clases con pocos datos reales (especialmente acentuadas) reciben más peso en loss.
# Esto complementa el WeightedRandomSampler (que balancea sampling)
# con un loss que penaliza más los errores en clases difíciles.

train_meta = meta[meta['split'] == 'train']
train_class_counts = np.bincount(
    train_meta['class_idx'].values, minlength=NUM_CLASSES
).clip(min=1).astype(np.float32)

# Effective number of samples (método de Cui et al. 2019)
# Maneja mejor el desbalance que simple inverse frequency
beta = 0.999
effective_num = 1.0 - np.power(beta, train_class_counts)
class_weights_np = (1.0 - beta) / effective_num

# Normalizar para que el promedio sea 1.0
class_weights_np = class_weights_np / class_weights_np.mean()

# Boost extra para clases acentuadas (las más problemáticas)
ACCENT_CHARS_ALL = set()
for _, acc_list in ACCENT_MAP.items():
    for acc_char, _ in acc_list:
        acc_idx = char_to_idx(acc_char)
        if acc_idx is not None:
            ACCENT_CHARS_ALL.add(acc_idx)

ACCENT_BOOST = 1.5  # 50% más peso en loss para acentuadas
for idx in ACCENT_CHARS_ALL:
    class_weights_np[idx] *= ACCENT_BOOST

class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)

print('=== Class weights para FocalLoss ===')
print(f'Min weight:  {class_weights_np.min():.3f} '
      f'({IDX2CHAR[class_weights_np.argmin()]})')
print(f'Max weight:  {class_weights_np.max():.3f} '
      f'({IDX2CHAR[class_weights_np.argmax()]})')
print(f'Mean weight: {class_weights_np.mean():.3f}')

# Mostrar weights de clases acentuadas
print(f'\nWeights de clases acentuadas (boosted x{ACCENT_BOOST}):')
for idx in sorted(ACCENT_CHARS_ALL):
    print(f'  {IDX2CHAR[idx]!r:4s} (idx={idx:3d}): weight={class_weights_np[idx]:.3f}')

# ── Train config ─────────────────────────────────────────────────────────────
train_config = {
    'run_id': RUN_ID,
    'model': 'tf_efficientnetv2_s + ProjectionHead + ArcFace v5',
    'backbone': 'tf_efficientnetv2_s',
    'embed_dim': EMBED_DIM,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'max_epochs': MAX_EPOCHS,
    'freeze_epochs': FREEZE_EPOCHS,
    'patience': PATIENCE,
    'lr_head': LR_HEAD,
    'lr_backbone': LR_BACKBONE,
    'weight_decay': WEIGHT_DECAY,
    'dropout_rate': DROPOUT_RATE,
    'label_smoothing': LABEL_SMOOTH,
    'mixup_alpha': MIXUP_ALPHA,
    'tta_n': TTA_N,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR (sin restarts)',
    'warmup_epochs': 3,
    'scheduler_T_max': MAX_EPOCHS - FREEZE_EPOCHS,
    'scheduler_eta_min': 1e-6,
    'grad_clip_max_norm': 1.0,
    'loss': 'FocalLoss(gamma=2.0) + class_weights + label_smoothing',
    'arcface_s': ARCFACE_S,
    'arcface_m': ARCFACE_M,
    'mixed_precision': True,
    'letterbox_resize': True,
    'accent_augmentation': True,
    'source_weighted_sampling': True,
    'class_weighted_loss': True,
    'accent_boost_in_loss': ACCENT_BOOST,
    'changes_vs_v4': [
        'Projection Head 1280→512 con BN+ReLU+Dropout',
        'Accent Augmentation desde EMNIST real (400/clase)',
        'EMNIST_MAX_PER_CLASS 500→800',
        'SYNTH_PER_CLASS 100→500',
        'FREEZE_EPOCHS 2→5 (estabilizar projector+ArcFace)',
        'LR_HEAD 1e-2→5e-3 (menos agresivo)',
        'LR_BACKBONE 2e-4→1e-4 (fine-tune conservador)',
        'DROPOUT 0.5→0.4',
        'CosineAnnealingLR sin restarts (más estable)',
        'Warmup 3 epochs (LR crece linealmente)',
        'FocalLoss con class_weights (effective number)',
        'Accent classes boosted 1.5x en loss',
        'Source-weighted sampling (verack 1.5x, accent_aug 1.3x)',
        'Test set honesto: solo datos realistas para clases con datos reales',
        'Hard pairs incluyen pares acentuados',
        'ElasticTransform más fuerte en augmentaciones',
        'Morphological ops para simular grosor de trazo',
        'TTA 5 augmentaciones (añade elastic)',
    ],
    'seed': SEED,
}

with open(TRAIN_CFG_PATH, 'w') as f:
    json.dump(train_config, f, indent=2)
print('\nGuardado train_config.json')

# ── Resumen visual de la configuración ───────────────────────────────────────
print(f'\n{"="*60}')
print(f'=== CONFIGURACIÓN DE ENTRENAMIENTO v5 ===')
print(f'{"="*60}')
print(f'Modelo:     EfficientNetV2-S → Proj({EMBED_DIM}) → ArcFace({NUM_CLASSES})')
print(f'Epochs:     {MAX_EPOCHS} (freeze {FREEZE_EPOCHS} + warmup 3)')
print(f'LR:         head={LR_HEAD} backbone={LR_BACKBONE}')
print(f'Loss:       FocalLoss(γ=2.0) + class_weights + LS={LABEL_SMOOTH}')
print(f'Scheduler:  CosineAnnealingLR (T_max={MAX_EPOCHS - FREEZE_EPOCHS})')
print(f'ArcFace:    s={ARCFACE_S} m={ARCFACE_M}')
print(f'Dropout:    {DROPOUT_RATE}')
print(f'Mixup:      α={MIXUP_ALPHA}')
print(f'Timer:      {MAX_TRAINING_MINUTES} min max')
print(f'{"="*60}')

## Cell 13 — Phase B: Training loop con Mixup + val sin margin

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13 — Phase B: Training loop v5 (Warmup + FocalLoss + Stable scheduler)
# ══════════════════════════════════════════════════════════════════════════════

# ── Loss con class weights ───────────────────────────────────────────────────
criterion = FocalLoss(
    gamma=2.0,
    label_smoothing=LABEL_SMOOTH,
    class_weights=class_weights_tensor
)

scaler = GradScaler()

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1: Freeze backbone, train only projector + ArcFace
# ══════════════════════════════════════════════════════════════════════════════

# Congelar backbone
for param in model.backbone.parameters():
    param.requires_grad = False

# Solo entrenar projector + arcface
optimizer = torch.optim.AdamW([
    {'params': model.projector.parameters(), 'lr': LR_HEAD, 'name': 'projector'},
    {'params': model.arcface.parameters(),   'lr': LR_HEAD, 'name': 'arcface'},
], weight_decay=WEIGHT_DECAY)

# Scheduler: se crea después del unfreeze (epoch FREEZE_EPOCHS+1)
scheduler = None


def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # Asegurar lam >= 0.5 para estabilidad
    bs = images.size(0)
    perm = torch.randperm(bs, device=images.device)
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam


def get_lr_string(opt):
    """Muestra LR de cada param group."""
    parts = []
    for i, pg in enumerate(opt.param_groups):
        name = pg.get('name', f'group{i}')
        parts.append(f'{name}={pg["lr"]:.1e}')
    return ' | '.join(parts)


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def train_epoch(epoch_num: int):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Mixup
        mixed, la, lb, lam = mixup_batch(images, labels)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(mixed, la)
            loss = lam * criterion(logits, la) + (1 - lam) * criterion(logits, lb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        total_correct += (lam * (preds == la).sum().item() +
                         (1 - lam) * (preds == lb).sum().item())
        total_loss += loss.item() * len(la)
        total += len(la)

    return total_loss / max(total, 1), total_correct / max(total, 1)


def val_epoch():
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    val_ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(images)  # inference sin margin
                loss = val_ce(logits, labels)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_loss += loss.item() * len(labels)
            total += len(labels)

    return total_loss / max(total, 1), total_correct / max(total, 1)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': [], 'phase': []
}
best_val_acc = 0.0
best_epoch   = 0
patience_ctr = 0
train_start  = time.time()

print(f'=== Entrenando v5 ===')
print(f'Phase 1: Freeze backbone, train projector+ArcFace '
      f'(epochs 1-{FREEZE_EPOCHS})')
print(f'Phase 2: Unfreeze backbone + warmup + cosine annealing '
      f'(epochs {FREEZE_EPOCHS+1}-{MAX_EPOCHS})')
print()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    elapsed_total_min = (time.time() - train_start) / 60

    # ── Timer check ──────────────────────────────────────────────────────
    if elapsed_total_min > MAX_TRAINING_MINUTES:
        print(f'\n  ⏰ Timer: {elapsed_total_min:.0f}min — deteniendo')
        break

    # ══════════════════════════════════════════════════════════════════════
    # PHASE TRANSITION: Unfreeze backbone at epoch FREEZE_EPOCHS+1
    # ══════════════════════════════════════════════════════════════════════
    if epoch == FREEZE_EPOCHS + 1:
        print(f'\n{"="*60}')
        print(f'Phase 2: Descongelando backbone + warmup')
        print(f'{"="*60}')

        # Descongelar backbone
        for param in model.backbone.parameters():
            param.requires_grad = True

        # Dividir backbone en 3 grupos por profundidad
        backbone_params = list(model.backbone.named_parameters())
        n = len(backbone_params)
        third = n // 3

        early_params  = [p for i, (_, p) in enumerate(backbone_params) if i < third]
        middle_params = [p for i, (_, p) in enumerate(backbone_params) if third <= i < 2*third]
        late_params   = [p for i, (_, p) in enumerate(backbone_params) if i >= 2*third]

        print(f'  Backbone dividido: early={len(early_params)}, '
              f'middle={len(middle_params)}, late={len(late_params)}')

        # Nuevo optimizer con todos los param groups
        # LR inicial bajo (warmup lo subirá)
        warmup_factor = 0.1  # empezar con 10% del LR target
        optimizer = torch.optim.AdamW([
            {'params': model.projector.parameters(),
             'lr': LR_HEAD * warmup_factor, 'name': 'projector'},
            {'params': model.arcface.parameters(),
             'lr': LR_HEAD * warmup_factor, 'name': 'arcface'},
            {'params': early_params,
             'lr': LR_BACKBONE * 0.1 * warmup_factor, 'name': 'backbone_early'},
            {'params': middle_params,
             'lr': LR_BACKBONE * 0.3 * warmup_factor, 'name': 'backbone_mid'},
            {'params': late_params,
             'lr': LR_BACKBONE * warmup_factor, 'name': 'backbone_late'},
        ], weight_decay=WEIGHT_DECAY)

        # Target LRs para después del warmup
        target_lrs = [
            LR_HEAD,                # projector
            LR_HEAD,                # arcface
            LR_BACKBONE * 0.1,      # backbone early
            LR_BACKBONE * 0.3,      # backbone mid
            LR_BACKBONE,            # backbone late
        ]

        # Scheduler: cosine annealing sin restarts
        T_max = MAX_EPOCHS - FREEZE_EPOCHS - 3  # -3 para warmup
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(T_max, 10), eta_min=1e-6
        )

        print(f'  Scheduler: CosineAnnealingLR T_max={max(T_max, 10)}')
        print(f'  Warmup: 3 epochs linearly to target LRs')
        print()

    # ══════════════════════════════════════════════════════════════════════
    # WARMUP: Linear LR increase for first 3 epochs after unfreeze
    # ══════════════════════════════════════════════════════════════════════
    warmup_epochs = 3
    if FREEZE_EPOCHS < epoch <= FREEZE_EPOCHS + warmup_epochs:
        warmup_step = epoch - FREEZE_EPOCHS  # 1, 2, 3
        warmup_progress = warmup_step / warmup_epochs  # 0.33, 0.67, 1.0
        for pg_idx, pg in enumerate(optimizer.param_groups):
            pg['lr'] = target_lrs[pg_idx] * warmup_progress

    # ── Determine phase ──────────────────────────────────────────────────
    if epoch <= FREEZE_EPOCHS:
        phase = 'frozen'
    elif epoch <= FREEZE_EPOCHS + warmup_epochs:
        phase = 'warmup'
    else:
        phase = 'full'

    # ── Train + Validate ─────────────────────────────────────────────────
    tr_loss, tr_acc = train_epoch(epoch)
    vl_loss, vl_acc = val_epoch()

    # Step scheduler (only after warmup)
    if scheduler is not None and epoch > FREEZE_EPOCHS + warmup_epochs:
        scheduler.step()

    elapsed = time.time() - t0
    total_min = (time.time() - train_start) / 60

    # ── Record history ───────────────────────────────────────────────────
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['phase'].append(phase)

    # ── Print ────────────────────────────────────────────────────────────
    lr_str = get_lr_string(optimizer)
    print(f'Epoch {epoch:3d}/{MAX_EPOCHS} [{phase:6s}]  '
          f'tr={tr_loss:.3f}/{tr_acc:.3f}  vl={vl_loss:.3f}/{vl_acc:.3f}  '
          f'{elapsed:.0f}s [{total_min:.0f}m]')
    if epoch <= FREEZE_EPOCHS + warmup_epochs + 1 or epoch % 5 == 0:
        print(f'  LR: {lr_str}')

    # ── Checkpoint ───────────────────────────────────────────────────────
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_epoch = epoch
        patience_ctr = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'char_map': IDX2CHAR,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'embed_dim': EMBED_DIM,
            'backbone': 'tf_efficientnetv2_s',
            'arcface_s': ARCFACE_S,
            'arcface_m': ARCFACE_M,
            'projection_head': True,
            'metrics': {
                'val_acc': best_val_acc,
                'best_epoch': best_epoch,
                'phase': phase,
            }
        }, BEST_PT_PATH)
        print(f'  ✓ Mejor (val_acc={best_val_acc:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  Early stopping (patience={PATIENCE})')
            break

training_time_min = (time.time() - train_start) / 60
print(f'\n{"="*60}')
print(f'Entrenamiento completo')
print(f'  best_val_acc = {best_val_acc:.4f} @ epoch {best_epoch}')
print(f'  Tiempo total = {training_time_min:.0f} min')
print(f'  Epochs completados = {len(history["train_loss"])}')
print(f'{"="*60}')

## Cell 14 — Phase B: Curvas de entrenamiento

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14 — Phase B: Curvas de entrenamiento con fases visualizadas
# ══════════════════════════════════════════════════════════════════════════════

n_epochs = len(history['train_loss'])
ep = list(range(1, n_epochs + 1))

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# ── Colores de fase ──────────────────────────────────────────────────────────
phase_colors = {
    'frozen': '#FFE0B2',   # naranja claro
    'warmup': '#C8E6C9',   # verde claro
    'full':   '#BBDEFB',   # azul claro
}

def add_phase_background(ax, epochs, phases):
    """Añade fondo coloreado por fase al subplot."""
    if not phases:
        return
    prev_phase = phases[0]
    start = epochs[0]
    for i in range(1, len(phases)):
        if phases[i] != prev_phase or i == len(phases) - 1:
            end = epochs[i] if phases[i] != prev_phase else epochs[i]
            color = phase_colors.get(prev_phase, '#FFFFFF')
            ax.axvspan(start - 0.5, end + 0.5, alpha=0.3, color=color)
            prev_phase = phases[i]
            start = epochs[i]
    # Último tramo
    color = phase_colors.get(prev_phase, '#FFFFFF')
    ax.axvspan(start - 0.5, epochs[-1] + 0.5, alpha=0.3, color=color)

phases = history.get('phase', ['full'] * n_epochs)

# ── 1. Train Loss ────────────────────────────────────────────────────────────
ax = axes[0, 0]
add_phase_background(ax, ep, phases)
ax.plot(ep, history['train_loss'], 'b-', linewidth=1.5, label='Train Loss')
ax.set_title('Train Loss', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
ax.legend()

# ── 2. Val Loss ──────────────────────────────────────────────────────────────
ax = axes[0, 1]
add_phase_background(ax, ep, phases)
ax.plot(ep, history['val_loss'], 'orange', linewidth=1.5, label='Val Loss')
ax.set_title('Val Loss (sin ArcFace margin)', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
ax.legend()

# ── 3. Train Accuracy ───────────────────────────────────────────────────────
ax = axes[1, 0]
add_phase_background(ax, ep, phases)
ax.plot(ep, history['train_acc'], 'b-', linewidth=1.5, label='Train Acc')
ax.set_title('Train Accuracy (con mixup — subestima)', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.grid(True, alpha=0.3)
ax.legend()

# ── 4. Val Accuracy ─────────────────────────────────────────────────────────
ax = axes[1, 1]
add_phase_background(ax, ep, phases)
ax.plot(ep, history['val_acc'], 'orange', linewidth=1.5, label='Val Acc')
ax.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7,
           label=f'Mejor ep {best_epoch} ({best_val_acc:.4f})')
ax.axhline(y=0.80, color='gray', linestyle=':', alpha=0.5, label='80% target')
ax.set_title('Val Accuracy (sin ArcFace margin)', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

# ── 5. Learning Rate ────────────────────────────────────────────────────────
ax = axes[2, 0]
add_phase_background(ax, ep, phases)
ax.plot(ep, history['lr'], 'g-', linewidth=1.5, label='LR (head/projector)')
ax.set_title('Learning Rate', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('LR')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend()

# ── 6. Leyenda de fases ────────────────────────────────────────────────────
ax = axes[2, 1]
ax.axis('off')

# Crear leyenda visual de fases
legend_items = [
    ('Frozen (projector+ArcFace only)', phase_colors['frozen']),
    ('Warmup (LR ramp-up)', phase_colors['warmup']),
    ('Full training (cosine annealing)', phase_colors['full']),
]

y_start = 0.85
for i, (label, color) in enumerate(legend_items):
    y = y_start - i * 0.15
    ax.add_patch(plt.Rectangle((0.05, y - 0.04), 0.08, 0.08,
                                facecolor=color, edgecolor='gray', alpha=0.7,
                                transform=ax.transAxes))
    ax.text(0.18, y, label, transform=ax.transAxes, fontsize=11,
            verticalalignment='center')

# Resumen de entrenamiento
summary_text = (
    f'Resumen:\n'
    f'  Best val_acc: {best_val_acc:.4f} @ epoch {best_epoch}\n'
    f'  Total epochs: {n_epochs}\n'
    f'  Freeze epochs: {FREEZE_EPOCHS}\n'
    f'  Warmup epochs: 3\n'
    f'  Training time: {training_time_min:.0f} min\n'
    f'  Patience: {PATIENCE}'
)
ax.text(0.05, 0.25, summary_text, transform=ax.transAxes, fontsize=10,
        fontfamily='monospace', verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))

plt.suptitle(f'Training Curves v5 — {RUN_ID}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=150)
plt.close()
print(f'Guardado {CURVES_PATH}')

# ── Resumen numérico ─────────────────────────────────────────────────────────
print(f'\n=== Resumen de entrenamiento ===')
print(f'  Epochs totales: {n_epochs}')

# Mostrar mejora por fase
for phase_name in ['frozen', 'warmup', 'full']:
    phase_epochs = [i for i, p in enumerate(phases) if p == phase_name]
    if not phase_epochs:
        continue
    phase_val_accs = [history['val_acc'][i] for i in phase_epochs]
    print(f'  {phase_name:8s}: epochs {phase_epochs[0]+1}-{phase_epochs[-1]+1}, '
          f'val_acc {phase_val_accs[0]:.4f} → {phase_val_accs[-1]:.4f}')

print(f'  Best: val_acc={best_val_acc:.4f} @ epoch {best_epoch}')

## Cell 15 — Phase C: Evaluación con TTA

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 15 — Phase C: Evaluación con TTA + métricas honestas
# ══════════════════════════════════════════════════════════════════════════════

# ── Cargar mejor checkpoint ──────────────────────────────────────────────────
ckpt = torch.load(BEST_PT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Checkpoint cargado: epoch {ckpt["metrics"]["best_epoch"]}, '
      f'val_acc={ckpt["metrics"]["val_acc"]:.4f}')


def predict_with_tta(df_split: pd.DataFrame,
                     tta_tfms: List) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Test-Time Augmentation: promedia probabilidades de N augmentaciones.
    """
    all_labels = df_split['class_idx'].values
    n_samples = len(all_labels)
    avg_probs = np.zeros((n_samples, NUM_CLASSES), dtype=np.float32)

    for t_idx, tfm in enumerate(tta_tfms):
        print(f'  TTA {t_idx + 1}/{len(tta_tfms)}...')
        tta_ds = CharDataset(df_split, tfm, use_letterbox=True)
        tta_loader = DataLoader(
            tta_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True
        )
        probs_list = []
        with torch.no_grad():
            for images, _ in tta_loader:
                images = images.to(DEVICE)
                with autocast():
                    logits = model(images)  # sin margin
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                probs_list.append(probs)
        avg_probs += np.vstack(probs_list)

    avg_probs /= len(tta_tfms)
    preds = avg_probs.argmax(axis=1)
    return preds, all_labels, avg_probs


# ══════════════════════════════════════════════════════════════════════════════
# EVALUACIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

print(f'=== Evaluando en test set con TTA x{TTA_N} ===')
all_preds, all_labels, all_probs = predict_with_tta(
    df_test, tta_transforms[:TTA_N]
)

test_acc = (all_preds == all_labels).mean()
print(f'\nTest accuracy global (TTA x{TTA_N}): {test_acc:.4f}')

# ── Classification report ────────────────────────────────────────────────────
target_names = [IDX2CHAR[i] for i in range(NUM_CLASSES)]
report = classification_report(
    all_labels, all_preds,
    labels=list(range(NUM_CLASSES)),
    target_names=target_names,
    output_dict=True, zero_division=0
)

per_class_f1 = {IDX2CHAR[i]: report.get(IDX2CHAR[i], {}).get('f1-score', 0.0)
                for i in range(NUM_CLASSES)}
per_class_acc = {}
for idx in range(NUM_CLASSES):
    mask = (all_labels == idx)
    per_class_acc[IDX2CHAR[idx]] = float((all_preds[mask] == idx).mean()) \
                                   if mask.sum() > 0 else 0.0

val_f1 = report['weighted avg']['f1-score']
print(f'Weighted F1: {val_f1:.4f}')


# ══════════════════════════════════════════════════════════════════════════════
# MÉTRICAS HONESTAS: separar clases reales vs solo sintéticas
# Esta es la métrica que correlaciona con el test real ("carpet")
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n{"="*60}')
print(f'=== MÉTRICAS HONESTAS (separadas por tipo de datos) ===')
print(f'{"="*60}')

# Obtener sources del test set
test_sources = df_test['source'].values

# ── 1. Accuracy en clases con datos REALES ───────────────────────────────────
# Estas clases tienen test con datos reales (EMNIST/verack/accent_aug)
real_source_mask = np.isin(test_sources, list(REAL_SOURCES))
if real_source_mask.sum() > 0:
    real_preds = all_preds[real_source_mask]
    real_labels = all_labels[real_source_mask]
    real_test_acc = (real_preds == real_labels).mean()
    print(f'\n📊 Accuracy en datos REALES (EMNIST+verack+accent_aug):')
    print(f'   {real_test_acc:.4f} ({real_source_mask.sum()} imgs)')
    print(f'   → Esta métrica predice el rendimiento en "carpet"')
else:
    real_test_acc = 0.0
    print(f'\n⚠️ No hay datos reales en test set')

# ── 2. Accuracy en clases SOLO SINTÉTICAS ────────────────────────────────────
synth_source_mask = ~real_source_mask
if synth_source_mask.sum() > 0:
    synth_preds = all_preds[synth_source_mask]
    synth_labels = all_labels[synth_source_mask]
    synth_test_acc = (synth_preds == synth_labels).mean()
    print(f'\n📊 Accuracy en datos SINTÉTICOS (solo clases sin datos reales):')
    print(f'   {synth_test_acc:.4f} ({synth_source_mask.sum()} imgs)')
    print(f'   → Esta métrica está INFLADA (test y train son del mismo dominio)')
else:
    synth_test_acc = 0.0

# ── 3. Accuracy por fuente ───────────────────────────────────────────────────
print(f'\n📊 Accuracy por fuente de datos:')
for src in sorted(set(test_sources)):
    src_mask = test_sources == src
    if src_mask.sum() == 0:
        continue
    src_acc = (all_preds[src_mask] == all_labels[src_mask]).mean()
    n_imgs = src_mask.sum()
    n_cls = len(set(all_labels[src_mask]))
    print(f'   {src:20s}: {src_acc:.4f} ({n_imgs:5d} imgs, {n_cls:3d} clases)')


# ══════════════════════════════════════════════════════════════════════════════
# EVALUACIÓN DE ACENTUADAS (el problema principal)
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n{"="*60}')
print(f'=== EVALUACIÓN DETALLADA DE ACENTUADAS ===')
print(f'{"="*60}')

# Pares base → acentuada
accent_eval_pairs = [
    ('a', 'á'), ('e', 'é'), ('i', 'í'), ('o', 'ó'), ('u', 'ú'),
    ('u', 'ü'), ('n', 'ñ'),
    ('A', 'Á'), ('E', 'É'), ('I', 'Í'), ('O', 'Ó'), ('U', 'Ú'),
    ('N', 'Ñ'),
]

print(f'\n{"Clase":>6s} {"Acc":>6s} {"F1":>6s} {"N_test":>7s} {"Confundida con":>30s}')
print(f'{"-"*60}')

accent_accs = []
accent_confusion_details = []

for base_char, acc_char in accent_eval_pairs:
    acc_idx = char_to_idx(acc_char)
    base_idx = char_to_idx(base_char)
    if acc_idx is None:
        continue

    mask = all_labels == acc_idx
    n_test = mask.sum()

    if n_test == 0:
        print(f'{acc_char!r:>6s} {"N/A":>6s} {"N/A":>6s} {0:>7d} {"sin datos en test":>30s}')
        continue

    acc = (all_preds[mask] == acc_idx).mean()
    f1 = per_class_f1.get(acc_char, 0.0)
    accent_accs.append(acc)

    # ¿Con qué se confunde más?
    preds_for_this = all_preds[mask]
    wrong_mask = preds_for_this != acc_idx
    if wrong_mask.sum() > 0:
        wrong_preds = preds_for_this[wrong_mask]
        most_confused = Counter(wrong_preds).most_common(3)
        confused_str = ', '.join(
            f'{IDX2CHAR[idx]!r}({cnt})' for idx, cnt in most_confused
        )
    else:
        confused_str = 'ninguna ✅'

    # ¿Se confunde con su base?
    if base_idx is not None:
        confused_with_base = (preds_for_this == base_idx).sum()
        base_pct = confused_with_base / n_test * 100
        if base_pct > 10:
            confused_str += f' ⚠️{base_char!r}={base_pct:.0f}%'
    
    acc_str = f'{acc:.3f}'
    f1_str = f'{f1:.3f}'
    color_tag = '✅' if acc >= 0.8 else '⚠️' if acc >= 0.5 else '❌'
    print(f'{acc_char!r:>6s} {acc_str:>6s} {f1_str:>6s} {n_test:>7d} {confused_str:>30s} {color_tag}')
    
    accent_confusion_details.append({
        'char': acc_char, 'base': base_char,
        'accuracy': float(acc), 'f1': float(f1),
        'n_test': int(n_test), 'confused_with': confused_str
    })

if accent_accs:
    mean_accent_acc = np.mean(accent_accs)
    print(f'\n📊 Accuracy promedio de acentuadas: {mean_accent_acc:.4f}')
    print(f'   (antes era ~0% porque eran solo sintéticas tipográficas)')


# ══════════════════════════════════════════════════════════════════════════════
# EVALUACIÓN DE PARES CONFUSOS (case + shape)
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n{"="*60}')
print(f'=== EVALUACIÓN DE PARES CONFUSOS ===')
print(f'{"="*60}')

case_pairs = [
    ('c', 'C'), ('s', 'S'), ('o', 'O'), ('m', 'M'),
    ('f', 'F'), ('w', 'W'), ('u', 'U'), ('v', 'V'),
    ('p', 'P'), ('k', 'K'),
]
shape_pairs = [
    ('1', 'l'), ('O', '0'),
]

print(f'\n{"Par":>8s} {"Acc_1":>6s} {"Acc_2":>6s} {"1→2":>5s} {"2→1":>5s} {"Status":>8s}')
print(f'{"-"*45}')

for ch1, ch2 in case_pairs + shape_pairs:
    idx1 = char_to_idx(ch1)
    idx2 = char_to_idx(ch2)
    if idx1 is None or idx2 is None:
        continue

    # Accuracy de ch1
    mask1 = all_labels == idx1
    n1 = mask1.sum()
    acc1 = (all_preds[mask1] == idx1).mean() if n1 > 0 else 0
    conf_1to2 = (all_preds[mask1] == idx2).sum() if n1 > 0 else 0

    # Accuracy de ch2
    mask2 = all_labels == idx2
    n2 = mask2.sum()
    acc2 = (all_preds[mask2] == idx2).mean() if n2 > 0 else 0
    conf_2to1 = (all_preds[mask2] == idx1).sum() if n2 > 0 else 0

    status = '✅' if acc1 >= 0.8 and acc2 >= 0.8 else '⚠️' if acc1 >= 0.5 and acc2 >= 0.5 else '❌'
    print(f'{ch1!r}/{ch2!r:>5s} {acc1:>6.3f} {acc2:>6.3f} {conf_1to2:>5d} {conf_2to1:>5d} {status:>8s}')


# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ══════════════════════════════════════════════════════════════════════════════

classes_below_50 = [c for c, f1 in per_class_f1.items() if f1 < 0.5]
classes_below_80 = [c for c, f1 in per_class_f1.items() if f1 < 0.8]

print(f'\n{"="*60}')
print(f'=== RESUMEN DE EVALUACIÓN v5 ===')
print(f'{"="*60}')
print(f'Test accuracy global:              {test_acc:.4f}')
print(f'Test accuracy datos REALES:        {real_test_acc:.4f} ← métrica honesta')
if synth_source_mask.sum() > 0:
    print(f'Test accuracy datos SINTÉTICOS:    {synth_test_acc:.4f} ← inflada')
print(f'Weighted F1:                       {val_f1:.4f}')
if accent_accs:
    print(f'Accuracy promedio acentuadas:      {mean_accent_acc:.4f}')
print(f'Clases con F1 < 0.5:              {len(classes_below_50)} → {classes_below_50}')
print(f'Clases con F1 < 0.8:              {len(classes_below_80)}')
print(f'{"="*60}')

## Cell 16 — Phase C: Confusion matrix agrupada

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 16 — Phase C: Confusion matrices por grupo + accent cross-confusion
# ══════════════════════════════════════════════════════════════════════════════

cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))

# ── Definir grupos ───────────────────────────────────────────────────────────
lowercase_chars = list('abcdefghijklmnopqrstuvwxyz') + ['ñ']
uppercase_chars = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ') + ['Ñ']
digit_chars     = list('0123456789')
accented_lower  = list('áéíóúü')
accented_upper  = list('ÁÉÍÓÚÜ')
accented_chars  = accented_lower + accented_upper

# Acentuadas + sus bases juntas (para ver cross-confusion)
accent_cross_chars = []
accent_cross_pairs = [
    ('a', 'á'), ('e', 'é'), ('i', 'í'), ('o', 'ó'), ('u', 'ú'), ('u', 'ü'),
    ('n', 'ñ'),
    ('A', 'Á'), ('E', 'É'), ('I', 'Í'), ('O', 'Ó'), ('U', 'Ú'),
    ('N', 'Ñ'),
]
for base, acc in accent_cross_pairs:
    if base not in accent_cross_chars:
        accent_cross_chars.append(base)
    if acc not in accent_cross_chars:
        accent_cross_chars.append(acc)

# Case confusion chars
case_chars = []
case_pairs_viz = [
    ('c', 'C'), ('s', 'S'), ('o', 'O'), ('m', 'M'), ('f', 'F'),
    ('w', 'W'), ('u', 'U'), ('v', 'V'), ('p', 'P'), ('k', 'K'),
]
for ch1, ch2 in case_pairs_viz:
    if ch1 not in case_chars:
        case_chars.append(ch1)
    if ch2 not in case_chars:
        case_chars.append(ch2)

# Shape confusion chars
shape_chars = ['1', 'l', 'O', '0', 'I']

all_special = set(accent_cross_chars + case_chars + shape_chars +
                  accented_chars + lowercase_chars + uppercase_chars + digit_chars)
punct_chars = [c for c in IDX2CHAR.values() if c not in all_special]

# ── Grupos para las matrices ─────────────────────────────────────────────────
groups = [
    ('Minúsculas (27)',      lowercase_chars),
    ('Mayúsculas (27)',      uppercase_chars),
    ('Dígitos (10)',         digit_chars),
    ('Acentuadas vs Base',  accent_cross_chars),   # NUEVO: el grupo más importante
    ('Case Confusion',      case_chars),            # NUEVO
    ('Puntuación + Trazos', punct_chars),
]

# ══════════════════════════════════════════════════════════════════════════════
# Plotear confusion matrices
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for i, (gname, chars) in enumerate(groups):
    if i >= len(axes):
        break
    
    idxs = [CHAR2IDX[c] for c in chars if c in CHAR2IDX]
    if not idxs:
        axes[i].set_visible(False)
        continue
    
    sub_cm = cm[np.ix_(idxs, idxs)]
    row_sums = sub_cm.sum(axis=1, keepdims=True).clip(min=1)
    sub_cm_norm = sub_cm / row_sums
    
    # Decidir si mostrar anotaciones (para grupos pequeños)
    show_annot = len(idxs) <= 25
    fmt = '.2f' if show_annot else ''
    
    tick_labels = [IDX2CHAR[j] for j in idxs]
    
    # Colormap: destacar la diagonal (correcto) y los errores
    sns.heatmap(
        sub_cm_norm, ax=axes[i],
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        cmap='Blues', vmin=0, vmax=1,
        annot=show_annot, fmt=fmt,
        annot_kws={'size': 7} if show_annot else {},
        linewidths=0.5 if len(idxs) <= 25 else 0,
        cbar_kws={'shrink': 0.8}
    )
    
    # Calcular accuracy promedio del grupo
    diag = np.diag(sub_cm_norm)
    group_acc = diag.mean()
    group_min = diag.min()
    min_char = tick_labels[diag.argmin()]
    
    axes[i].set_title(
        f'{gname}\n'
        f'Avg acc: {group_acc:.3f} | Min: {min_char!r}={group_min:.3f}',
        fontsize=11
    )
    axes[i].set_xlabel('Predicho', fontsize=9)
    axes[i].set_ylabel('Real', fontsize=9)
    axes[i].tick_params(axis='both', labelsize=7)

# Ocultar ejes sobrantes
for i in range(len(groups), len(axes)):
    axes[i].set_visible(False)

plt.suptitle(
    f'Confusion Matrices por Grupo (Normalizada) — Test acc={test_acc:.4f}',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(CONFMAT_PATH, dpi=150)
plt.close()
print(f'Guardado {CONFMAT_PATH}')


# ══════════════════════════════════════════════════════════════════════════════
# NUEVO: Confusion matrix detallada SOLO de acentuadas vs su base
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n=== Confusion Matrix: Acentuadas vs Base (detalle) ===')

fig2, axes2 = plt.subplots(2, 4, figsize=(24, 10))
axes2 = axes2.flatten()

detail_pairs = [
    ('a/á', ['a', 'á']),
    ('e/é', ['e', 'é']),
    ('i/í', ['i', 'í']),
    ('o/ó', ['o', 'ó']),
    ('u/ú/ü', ['u', 'ú', 'ü']),
    ('n/ñ', ['n', 'ñ']),
    ('A-Z accent', ['A', 'Á', 'E', 'É', 'I', 'Í']),
    ('O-Z accent', ['O', 'Ó', 'U', 'Ú', 'N', 'Ñ']),
]

for ax_i, (pair_name, chars) in enumerate(detail_pairs):
    if ax_i >= len(axes2):
        break
    
    idxs = [CHAR2IDX[c] for c in chars if c in CHAR2IDX]
    if not idxs:
        axes2[ax_i].set_visible(False)
        continue
    
    sub_cm = cm[np.ix_(idxs, idxs)]
    row_sums = sub_cm.sum(axis=1, keepdims=True).clip(min=1)
    sub_cm_norm = sub_cm / row_sums
    
    tick_labels = [IDX2CHAR[j] for j in idxs]
    
    sns.heatmap(
        sub_cm_norm, ax=axes2[ax_i],
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        cmap='RdYlGn_r', vmin=0, vmax=1,
        annot=True, fmt='.2f',
        annot_kws={'size': 10, 'fontweight': 'bold'},
        linewidths=1,
        cbar=False
    )
    
    diag = np.diag(sub_cm_norm)
    acc = diag.mean()
    status = '✅' if acc >= 0.8 else '⚠️' if acc >= 0.5 else '❌'
    
    axes2[ax_i].set_title(f'{pair_name} (acc={acc:.3f}) {status}', fontsize=11)
    axes2[ax_i].set_xlabel('Predicho', fontsize=9)
    axes2[ax_i].set_ylabel('Real', fontsize=9)

for i in range(len(detail_pairs), len(axes2)):
    axes2[i].set_visible(False)

plt.suptitle(
    'Accent Confusion Detail — ¿Se confunden base↔acentuada?',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
accent_confmat_path = WORK / 'accent_confusion_detail.png'
plt.savefig(accent_confmat_path, dpi=150)
plt.close()
print(f'Guardado {accent_confmat_path}')


# ══════════════════════════════════════════════════════════════════════════════
# Top-10 pares confusos con clasificación de tipo
# ══════════════════════════════════════════════════════════════════════════════

cm_nodiag = cm.copy()
np.fill_diagonal(cm_nodiag, 0)

# Clasificar tipo de confusión
def classify_confusion(true_char: str, pred_char: str) -> str:
    """Clasifica el tipo de confusión entre dos caracteres."""
    # Accent confusion
    accent_bases = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
                    'ü': 'u', 'ñ': 'n',
                    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
                    'Ü': 'U', 'Ñ': 'N'}
    if true_char in accent_bases and pred_char == accent_bases[true_char]:
        return 'accent→base'
    if pred_char in accent_bases and true_char == accent_bases[pred_char]:
        return 'base→accent'
    
    # Case confusion
    if true_char.isalpha() and pred_char.isalpha():
        if true_char.lower() == pred_char.lower() and true_char != pred_char:
            return 'case'
    
    # Shape confusion
    shape_groups = [{'1', 'l', 'I'}, {'0', 'O', 'o'}, {'5', 'S', 's'}]
    for group in shape_groups:
        if true_char in group and pred_char in group:
            return 'shape'
    
    return 'other'


flat = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_nodiag[i, j] > 0:
            flat.append((
                cm_nodiag[i, j],
                IDX2CHAR[i],
                IDX2CHAR[j],
                classify_confusion(IDX2CHAR[i], IDX2CHAR[j])
            ))

flat.sort(reverse=True)

top10_pairs = []
for count, true_ch, pred_ch, conf_type in flat[:15]:  # top 15 para más info
    top10_pairs.append({
        'true': true_ch,
        'pred': pred_ch,
        'count': int(count),
        'type': conf_type
    })

with open(TOP10_PATH, 'w', encoding='utf-8') as f:
    json.dump(top10_pairs, f, indent=2, ensure_ascii=False)

print(f'\nTop-15 pares confusos (con tipo):')
print(f'{"True":>6s} {"→":>2s} {"Pred":>6s} {"Count":>6s} {"Tipo":>15s}')
print(f'{"-"*40}')
for item in top10_pairs:
    print(f'{item["true"]!r:>6s}  → {item["pred"]!r:>6s} {item["count"]:>6d} '
          f'{item["type"]:>15s}')

# Resumen por tipo de confusión
type_counts = Counter(item['type'] for item in top10_pairs)
print(f'\nResumen de tipos de confusión (top-15):')
for ctype, cnt in type_counts.most_common():
    print(f'  {ctype:>15s}: {cnt}')

## Cell 17 — Phase C: Predicciones de muestra

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 17 — Phase C: Predicciones de muestra (dirigidas + aleatorias)
# ══════════════════════════════════════════════════════════════════════════════

mean_ = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def predict_single(ds, idx):
    """Predice una imagen y devuelve detalles."""
    tensor, gt_label = ds[idx]
    with torch.no_grad():
        logit = model(tensor.unsqueeze(0).to(DEVICE))
        prob = torch.softmax(logit, dim=1).cpu().squeeze()
        pred_idx = int(prob.argmax())
        confidence = float(prob[pred_idx])
    
    # Desnormalizar para visualización
    img_show = (tensor * std_ + mean_).permute(1, 2, 0).clamp(0, 1).numpy()
    
    return {
        'image': img_show,
        'gt_label': gt_label,
        'gt_char': IDX2CHAR[gt_label],
        'pred_idx': pred_idx,
        'pred_char': IDX2CHAR[pred_idx],
        'confidence': confidence,
        'correct': gt_label == pred_idx,
    }


# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 1: Muestras DIRIGIDAS — acentuadas y pares confusos
# ══════════════════════════════════════════════════════════════════════════════

# Seleccionar indices del test set para caracteres específicos
df_test_reset = df_test.reset_index(drop=True)

target_chars = [
    # Pares acentuados (base + acentuada)
    'a', 'á', 'e', 'é', 'i', 'í', 'o', 'ó', 'u', 'ú',
    'n', 'ñ', 'N', 'Ñ',
    # Pares confusos case
    'c', 'C', 's', 'S',
    # Shape confusion
    '1', 'l',
]

directed_indices = []
for ch in target_chars:
    ch_idx = char_to_idx(ch)
    if ch_idx is None:
        continue
    matching = df_test_reset[df_test_reset['class_idx'] == ch_idx].index.tolist()
    if matching:
        # Elegir uno aleatorio
        directed_indices.append(random.choice(matching))

# Limitar a 20 muestras
directed_indices = directed_indices[:20]

if directed_indices:
    n_samples = len(directed_indices)
    n_cols = 5
    n_rows = math.ceil(n_samples / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.5 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()
    
    for ax_i, ds_i in enumerate(directed_indices):
        result = predict_single(test_ds, ds_i)
        source = df_test_reset.iloc[ds_i]['source']
        
        axes[ax_i].imshow(result['image'])
        
        color = 'green' if result['correct'] else 'red'
        
        # Borde coloreado
        for spine in axes[ax_i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        
        title = (
            f"GT: {result['gt_char']!r}  Pred: {result['pred_char']!r}\n"
            f"Conf: {result['confidence']:.2f} | {source}"
        )
        axes[ax_i].set_title(title, color=color, fontsize=8, fontweight='bold')
        axes[ax_i].axis('off')
    
    for i in range(len(directed_indices), len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle(
        'Predicciones Dirigidas: Acentuadas + Pares Confusos',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(WORK / 'directed_predictions.png', dpi=150)
    plt.close()
    print('Guardado directed_predictions.png')
    
    # Resumen
    n_correct = sum(1 for i in directed_indices if predict_single(test_ds, i)['correct'])
    print(f'  Dirigidas: {n_correct}/{len(directed_indices)} correctas '
          f'({100*n_correct/len(directed_indices):.0f}%)')


# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 2: Comparación lado a lado — base vs acentuada
# ══════════════════════════════════════════════════════════════════════════════

compare_pairs = [
    ('a', 'á'), ('e', 'é'), ('i', 'í'), ('o', 'ó'),
    ('u', 'ú'), ('n', 'ñ'), ('A', 'Á'), ('N', 'Ñ'),
]

fig2, axes2 = plt.subplots(len(compare_pairs), 2, figsize=(7, 3 * len(compare_pairs)))

for row, (base_ch, acc_ch) in enumerate(compare_pairs):
    for col, ch in enumerate([base_ch, acc_ch]):
        ch_idx = char_to_idx(ch)
        if ch_idx is None:
            axes2[row, col].set_visible(False)
            continue
        
        matching = df_test_reset[df_test_reset['class_idx'] == ch_idx].index.tolist()
        if not matching:
            axes2[row, col].text(0.5, 0.5, f'{ch!r}\nsin datos',
                                  ha='center', va='center',
                                  transform=axes2[row, col].transAxes, fontsize=10)
            axes2[row, col].axis('off')
            continue
        
        ds_i = random.choice(matching)
        result = predict_single(test_ds, ds_i)
        source = df_test_reset.iloc[ds_i]['source']
        
        axes2[row, col].imshow(result['image'])
        
        color = 'green' if result['correct'] else 'red'
        for spine in axes2[row, col].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        
        tag = '✅' if result['correct'] else '❌'
        title = (
            f"GT: {result['gt_char']!r} → Pred: {result['pred_char']!r} "
            f"{tag}\n{source} | conf={result['confidence']:.2f}"
        )
        axes2[row, col].set_title(title, fontsize=8, color=color, fontweight='bold')
        axes2[row, col].axis('off')

# Headers
axes2[0, 0].text(0.5, 1.15, 'BASE', ha='center', va='bottom',
                  transform=axes2[0, 0].transAxes, fontsize=12,
                  fontweight='bold', color='navy')
axes2[0, 1].text(0.5, 1.15, 'ACENTUADA', ha='center', va='bottom',
                  transform=axes2[0, 1].transAxes, fontsize=12,
                  fontweight='bold', color='darkred')

plt.suptitle(
    'Comparación Base vs Acentuada — ¿El modelo distingue el acento?',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(WORK / 'base_vs_accent_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado base_vs_accent_comparison.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 3: Muestras ALEATORIAS (como antes, para visión general)
# ══════════════════════════════════════════════════════════════════════════════

random_indices = random.sample(range(len(test_ds)), min(20, len(test_ds)))
fig3, axes3 = plt.subplots(4, 5, figsize=(16, 13))
axes3 = axes3.flatten()

for ax_i, ds_i in enumerate(random_indices):
    result = predict_single(test_ds, ds_i)
    source = df_test_reset.iloc[ds_i]['source'] if ds_i < len(df_test_reset) else '?'
    
    axes3[ax_i].imshow(result['image'])
    
    color = 'green' if result['correct'] else 'red'
    for spine in axes3[ax_i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
    
    title = (
        f"GT: {result['gt_char']!r}  Pred: {result['pred_char']!r}\n"
        f"Conf: {result['confidence']:.2f} | {source}"
    )
    axes3[ax_i].set_title(title, color=color, fontsize=8)
    axes3[ax_i].axis('off')

plt.suptitle('Predicciones Aleatorias (test set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK / 'sample_predictions.png', dpi=150)
plt.close()
print('Guardado sample_predictions.png')

# ── Resumen de las 3 figuras ─────────────────────────────────────────────────
print(f'\n=== Predicciones visualizadas ===')
print(f'  1. directed_predictions.png     — acentuadas + pares confusos')
print(f'  2. base_vs_accent_comparison.png — base vs acentuada lado a lado')
print(f'  3. sample_predictions.png        — aleatorias (visión general)')

## Cell 18 — Phase C: Accuracy por clase

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 18 — Phase C: Accuracy por clase con tipo de datos
# ══════════════════════════════════════════════════════════════════════════════

# ── Clasificar cada clase por tipo de datos en test ──────────────────────────
df_test_reset = df_test.reset_index(drop=True)

class_data_type = {}  # idx → 'real', 'accent_aug', 'synthetic'
for idx in range(NUM_CLASSES):
    ch = IDX2CHAR[idx]
    class_test = df_test_reset[df_test_reset['class_idx'] == idx]
    if len(class_test) == 0:
        class_data_type[idx] = 'no_data'
        continue
    sources = set(class_test['source'].unique())
    if 'emnist' in sources or 'verack' in sources:
        class_data_type[idx] = 'real'
    elif 'accent_aug' in sources:
        class_data_type[idx] = 'accent_aug'
    else:
        class_data_type[idx] = 'synthetic'

# Colores por tipo de datos
DATA_TYPE_COLORS = {
    'real':       {'high': '#2E7D32', 'mid': '#FF8F00', 'low': '#C62828'},   # verde/naranja/rojo
    'accent_aug': {'high': '#1565C0', 'mid': '#7B1FA2', 'low': '#AD1457'},   # azul/morado/rosa
    'synthetic':  {'high': '#78909C', 'mid': '#A1887F', 'low': '#E0E0E0'},   # grises (menos confiable)
    'no_data':    {'high': '#FFFFFF', 'mid': '#FFFFFF', 'low': '#FFFFFF'},
}

def get_bar_color(acc: float, dtype: str) -> str:
    """Color de barra basado en accuracy y tipo de datos."""
    colors = DATA_TYPE_COLORS.get(dtype, DATA_TYPE_COLORS['synthetic'])
    if acc >= 0.8:
        return colors['high']
    elif acc >= 0.5:
        return colors['mid']
    else:
        return colors['low']


# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 1: Accuracy por clase completa (107 clases)
# ══════════════════════════════════════════════════════════════════════════════

chars_sorted = [IDX2CHAR[i] for i in range(NUM_CLASSES)]
acc_sorted = [per_class_acc.get(c, 0.0) for c in chars_sorted]
dtypes_sorted = [class_data_type.get(i, 'no_data') for i in range(NUM_CLASSES)]
colors_sorted = [get_bar_color(a, d) for a, d in zip(acc_sorted, dtypes_sorted)]

fig, ax = plt.subplots(figsize=(22, 7))
bars = ax.bar(range(NUM_CLASSES), acc_sorted, color=colors_sorted, edgecolor='none')

ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(chars_sorted, rotation=90, fontsize=6)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy por Clase (test set, TTA) — coloreado por tipo de datos', fontsize=13)
ax.axhline(y=0.8, color='blue', linestyle='--', alpha=0.4, label='80% target')
ax.axhline(y=0.5, color='red', linestyle=':', alpha=0.3, label='50% mínimo')

# Leyenda
legend_patches = [
    mpatches.Patch(color='#2E7D32', label='Real (EMNIST/verack) ≥80%'),
    mpatches.Patch(color='#FF8F00', label='Real 50-80%'),
    mpatches.Patch(color='#C62828', label='Real <50%'),
    mpatches.Patch(color='#1565C0', label='Accent Aug ≥80%'),
    mpatches.Patch(color='#7B1FA2', label='Accent Aug 50-80%'),
    mpatches.Patch(color='#AD1457', label='Accent Aug <50%'),
    mpatches.Patch(color='#78909C', label='Synthetic ≥80% (inflado)'),
    mpatches.Patch(color='#A1887F', label='Synthetic 50-80% (inflado)'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=7, ncol=2)

# Marcar acentuadas con triángulo arriba de la barra
accent_indices = [char_to_idx(ch) for ch in 'áéíóúüñÁÉÍÓÚÜÑ' if char_to_idx(ch) is not None]
for ai in accent_indices:
    if ai < NUM_CLASSES:
        ax.annotate('▼', xy=(ai, min(acc_sorted[ai] + 0.03, 1.02)),
                    ha='center', va='bottom', fontsize=6, color='darkred')

ax.set_ylim(0, 1.08)
plt.tight_layout()
plt.savefig(WORK / 'per_class_accuracy.png', dpi=150)
plt.close()
print('Guardado per_class_accuracy.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 2: Accuracy SOLO de clases con datos reales (métrica honesta)
# ══════════════════════════════════════════════════════════════════════════════

real_classes = [i for i in range(NUM_CLASSES) if class_data_type[i] == 'real']
accent_classes = [i for i in range(NUM_CLASSES) if class_data_type[i] == 'accent_aug']
synth_classes = [i for i in range(NUM_CLASSES) if class_data_type[i] == 'synthetic']

fig2, axes2 = plt.subplots(1, 3, figsize=(24, 6),
                            gridspec_kw={'width_ratios': [3, 2, 2]})

# Panel 1: Clases con datos REALES
if real_classes:
    real_chars = [IDX2CHAR[i] for i in real_classes]
    real_accs = [per_class_acc.get(IDX2CHAR[i], 0.0) for i in real_classes]
    real_colors = ['#2E7D32' if a >= 0.8 else '#FF8F00' if a >= 0.5 else '#C62828'
                   for a in real_accs]
    
    axes2[0].bar(range(len(real_classes)), real_accs, color=real_colors)
    axes2[0].set_xticks(range(len(real_classes)))
    axes2[0].set_xticklabels(real_chars, rotation=90, fontsize=7)
    axes2[0].axhline(y=0.8, color='blue', linestyle='--', alpha=0.4)
    
    real_mean = np.mean(real_accs)
    n_above_80 = sum(1 for a in real_accs if a >= 0.8)
    axes2[0].set_title(
        f'Datos REALES ({len(real_classes)} clases)\n'
        f'Mean acc: {real_mean:.3f} | ≥80%: {n_above_80}/{len(real_classes)}',
        fontsize=11
    )
    axes2[0].set_ylabel('Accuracy')

# Panel 2: Clases ACCENT AUGMENTED
if accent_classes:
    acc_chars = [IDX2CHAR[i] for i in accent_classes]
    acc_accs = [per_class_acc.get(IDX2CHAR[i], 0.0) for i in accent_classes]
    acc_colors = ['#1565C0' if a >= 0.8 else '#7B1FA2' if a >= 0.5 else '#AD1457'
                  for a in acc_accs]
    
    axes2[1].bar(range(len(accent_classes)), acc_accs, color=acc_colors)
    axes2[1].set_xticks(range(len(accent_classes)))
    axes2[1].set_xticklabels(acc_chars, rotation=90, fontsize=8)
    axes2[1].axhline(y=0.8, color='blue', linestyle='--', alpha=0.4)
    
    acc_mean = np.mean(acc_accs)
    n_above_80_acc = sum(1 for a in acc_accs if a >= 0.8)
    axes2[1].set_title(
        f'ACCENT AUGMENTED ({len(accent_classes)} clases)\n'
        f'Mean acc: {acc_mean:.3f} | ≥80%: {n_above_80_acc}/{len(accent_classes)}',
        fontsize=11
    )
    axes2[1].set_ylabel('Accuracy')

# Panel 3: Clases SOLO SINTÉTICAS
if synth_classes:
    syn_chars = [IDX2CHAR[i] for i in synth_classes]
    syn_accs = [per_class_acc.get(IDX2CHAR[i], 0.0) for i in synth_classes]
    syn_colors = ['#78909C' if a >= 0.8 else '#A1887F' if a >= 0.5 else '#E0E0E0'
                  for a in syn_accs]
    
    axes2[2].bar(range(len(synth_classes)), syn_accs, color=syn_colors)
    axes2[2].set_xticks(range(len(synth_classes)))
    axes2[2].set_xticklabels(syn_chars, rotation=90, fontsize=7)
    axes2[2].axhline(y=0.8, color='blue', linestyle='--', alpha=0.4)
    
    syn_mean = np.mean(syn_accs)
    axes2[2].set_title(
        f'SOLO SINTÉTICAS ({len(synth_classes)} clases)\n'
        f'Mean acc: {syn_mean:.3f} (⚠️ inflado)',
        fontsize=11
    )
    axes2[2].set_ylabel('Accuracy')

plt.suptitle(
    f'Accuracy por Tipo de Datos — honest_acc={real_test_acc:.4f}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(WORK / 'per_class_accuracy_by_type.png', dpi=150)
plt.close()
print('Guardado per_class_accuracy_by_type.png')


# ══════════════════════════════════════════════════════════════════════════════
# Resumen numérico
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n=== Resumen de Accuracy por Tipo ===')

all_accs = list(per_class_acc.values())
print(f'\nGLOBAL ({NUM_CLASSES} clases):')
print(f'  Mean: {np.mean(all_accs):.4f}')
print(f'  ≥80%: {sum(1 for a in all_accs if a >= 0.8)}/{NUM_CLASSES}')
print(f'  <50%: {sum(1 for a in all_accs if a < 0.5)}/{NUM_CLASSES}')

if real_classes:
    print(f'\nDATOS REALES ({len(real_classes)} clases) ← métrica honesta:')
    print(f'  Mean: {np.mean(real_accs):.4f}')
    print(f'  ≥80%: {n_above_80}/{len(real_classes)}')
    worst_real = sorted(zip(real_accs, [IDX2CHAR[i] for i in real_classes]))[:5]
    print(f'  Peores 5: {[(c, f"{a:.3f}") for a, c in worst_real]}')

if accent_classes:
    print(f'\nACCENT AUGMENTED ({len(accent_classes)} clases):')
    print(f'  Mean: {np.mean(acc_accs):.4f}')
    print(f'  ≥80%: {n_above_80_acc}/{len(accent_classes)}')
    worst_acc = sorted(zip(acc_accs, [IDX2CHAR[i] for i in accent_classes]))[:5]
    print(f'  Peores 5: {[(c, f"{a:.3f}") for a, c in worst_acc]}')

if synth_classes:
    print(f'\nSOLO SINTÉTICAS ({len(synth_classes)} clases) ← inflado:')
    print(f'  Mean: {np.mean(syn_accs):.4f}')

## Cell 19 — Phase C: Actualizar checkpoint con métricas finales

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 19 — Phase C: Actualizar checkpoint con métricas finales completas
# ══════════════════════════════════════════════════════════════════════════════

# ── Calcular métricas de acentuadas ──────────────────────────────────────────
accent_test_acc = 0.0
if accent_classes:
    accent_mask = np.isin(all_labels, accent_classes)
    if accent_mask.sum() > 0:
        accent_test_acc = (all_preds[accent_mask] == all_labels[accent_mask]).mean()

# ── Actualizar checkpoint ────────────────────────────────────────────────────
ckpt = torch.load(BEST_PT_PATH, map_location='cpu')

ckpt['metrics'] = {
    # Métricas globales
    'val_acc':             float(best_val_acc),
    'val_f1':              float(val_f1),
    'test_acc':            float(test_acc),
    
    # Métricas honestas (NUEVAS — las más importantes)
    'real_test_acc':       float(real_test_acc),
    'synth_test_acc':      float(synth_test_acc),
    'accent_test_acc':     float(accent_test_acc),
    
    # Detalles
    'best_epoch':          best_epoch,
    'tta_n':               TTA_N,
    'training_time_min':   round(training_time_min, 2),
    'total_epochs':        len(history['train_loss']),
    
    # Distribución de clases por tipo
    'n_real_classes':      len(real_classes),
    'n_accent_classes':    len(accent_classes),
    'n_synth_classes':     len(synth_classes),
    
    # Peores clases (para diagnóstico rápido)
    'classes_below_50_f1': classes_below_50,
    'n_classes_below_80':  len(classes_below_80),
}

# Asegurar que el checkpoint tiene toda la info necesaria para inference
ckpt['char_map']        = IDX2CHAR
ckpt['num_classes']     = NUM_CLASSES
ckpt['img_size']        = IMG_SIZE
ckpt['embed_dim']       = EMBED_DIM
ckpt['backbone']        = 'tf_efficientnetv2_s'
ckpt['arcface_s']       = ARCFACE_S
ckpt['arcface_m']       = ARCFACE_M
ckpt['projection_head'] = True

torch.save(ckpt, BEST_PT_PATH)

# ── Resumen ──────────────────────────────────────────────────────────────────
print(f'Checkpoint actualizado con métricas finales')
print(f'\n{"="*50}')
print(f'=== MÉTRICAS EN CHECKPOINT ===')
print(f'{"="*50}')
print(f'  val_acc:           {best_val_acc:.4f}')
print(f'  test_acc (global): {test_acc:.4f}')
print(f'  ─────────────────────────────')
print(f'  real_test_acc:     {real_test_acc:.4f} ← predictor de "carpet"')
print(f'  accent_test_acc:   {accent_test_acc:.4f} ← mejora principal v5')
print(f'  synth_test_acc:    {synth_test_acc:.4f} ← inflado')
print(f'  ─────────────────────────────')
print(f'  best_epoch:        {best_epoch}')
print(f'  F1 < 0.5:          {len(classes_below_50)} clases')
print(f'  F1 < 0.8:          {len(classes_below_80)} clases')
print(f'  ─────────────────────────────')
print(f'  embed_dim:         {EMBED_DIM}')
print(f'  projection_head:   True')
print(f'{"="*50}')

# ── Verificar que el checkpoint se puede cargar correctamente ─────────────────
print(f'\n=== Verificación de checkpoint ===')
ckpt_verify = torch.load(BEST_PT_PATH, map_location='cpu')
model_verify = OCRClassifier(
    num_classes=ckpt_verify['num_classes'],
    embed_dim=ckpt_verify.get('embed_dim', EMBED_DIM)
)
model_verify.load_state_dict(ckpt_verify['model_state_dict'])
model_verify.eval()

# Quick forward test
with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    out = model_verify(dummy)
    assert out.shape == (1, NUM_CLASSES), f'Shape mismatch: {out.shape}'
    assert out.abs().max() > 5.0, f'ArcFace scale missing'

print(f'✅ Checkpoint se carga y ejecuta correctamente')
print(f'   Output shape: {out.shape}, range=[{out.min():.1f}, {out.max():.1f}]')

del model_verify, ckpt_verify  # liberar memoria

## Cell 20 — Phase C: Exportación ONNX

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 20 — Phase C: Exportación ONNX con Projection Head
# ══════════════════════════════════════════════════════════════════════════════

class ONNXExportWrapper(nn.Module):
    """
    Wrapper para export ONNX que incluye:
    backbone → pool → projector (Linear+BN+ReLU+Dropout) → normalize → W*s
    
    CAMBIO vs v4: incluye projection head completo.
    El projector se copia como módulo (no se reconstruye) para mantener
    los pesos de BN exactos.
    """
    def __init__(self, classifier: OCRClassifier):
        super().__init__()
        self.backbone = classifier.backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Copiar el projector completo (Linear + BN + ReLU + Dropout)
        # IMPORTANTE: copiar como módulo, no reconstruir, para que BN
        # tenga running_mean y running_var correctos
        self.projector = classifier.projector
        
        # Pesos de ArcFace
        self.W = nn.Parameter(classifier.arcface.W.data.clone())
        self.s = float(classifier.arcface.s)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # Backbone features
        x = self.backbone.forward_features(images)    # (B, 1280, H, W)
        x = self.pool(x)                               # (B, 1280, 1, 1)
        x = x.squeeze(-1).squeeze(-1)                  # (B, 1280)
        
        # Projection head (Linear → BN → ReLU → Dropout)
        x = self.projector(x)                           # (B, 512)
        
        # Cosine similarity * scale
        feat_norm = F.normalize(x, p=2, dim=1)         # (B, 512)
        w_norm = F.normalize(self.W, p=2, dim=1)       # (107, 512)
        logits = F.linear(feat_norm, w_norm) * self.s   # (B, 107)
        
        return logits


# ══════════════════════════════════════════════════════════════════════════════
# Preparar modelo para export
# ══════════════════════════════════════════════════════════════════════════════

# Cargar mejor checkpoint
ckpt = torch.load(BEST_PT_PATH, map_location='cpu')
model_cpu = OCRClassifier(
    num_classes=NUM_CLASSES,
    embed_dim=ckpt.get('embed_dim', EMBED_DIM)
)
model_cpu.load_state_dict(ckpt['model_state_dict'])
model_cpu.eval()  # CRÍTICO: poner en eval para que BN use running stats

# Crear wrapper
export_model = ONNXExportWrapper(model_cpu)
export_model.eval()  # CRÍTICO: eval mode para export

# ══════════════════════════════════════════════════════════════════════════════
# Verificar que wrapper reproduce exactamente los resultados de PyTorch
# ══════════════════════════════════════════════════════════════════════════════

print('=== Verificación PyTorch original vs Wrapper ===')
for bs in [1, 2, 4]:
    dummy = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        orig = model_cpu(dummy)
        wrap = export_model(dummy)
        diff = (orig - wrap).abs().max().item()
        print(f'  batch={bs}: max_diff={diff:.6f} '
              f'range=[{wrap.min():.1f}, {wrap.max():.1f}]')
        assert diff < 0.01, f'Diferencia demasiado grande: {diff}'

print('✅ Wrapper reproduce PyTorch correctamente')


# ══════════════════════════════════════════════════════════════════════════════
# Export ONNX con múltiples estrategias (fallback)
# ══════════════════════════════════════════════════════════════════════════════

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
export_success = False

# Estrategia 1: dynamic_shapes (PyTorch 2.x)
print('\n--- Estrategia 1: dynamic_shapes ---')
try:
    batch_dim = torch.export.Dim("batch", min=1, max=256)
    torch.onnx.export(
        export_model,
        (dummy_input,),
        str(BEST_ONNX_PATH),
        input_names=['images'],
        output_names=['logits'],
        dynamic_shapes={'images': {0: batch_dim}},
        opset_version=18,
        do_constant_folding=True,
    )
    print('✅ Exportado con dynamic_shapes')
    export_success = True
except Exception as e1:
    print(f'Falló: {str(e1)[:200]}')

# Estrategia 2: dynamo=False (legacy exporter)
if not export_success:
    print('\n--- Estrategia 2: dynamo=False (legacy) ---')
    try:
        torch.onnx.export(
            export_model,
            dummy_input,
            str(BEST_ONNX_PATH),
            input_names=['images'],
            output_names=['logits'],
            dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=17,
            do_constant_folding=True,
            dynamo=False,
        )
        print('✅ Exportado con legacy exporter')
        export_success = True
    except Exception as e2:
        print(f'Falló: {str(e2)[:200]}')

# Estrategia 3: jit.trace + export
if not export_success:
    print('\n--- Estrategia 3: jit.trace + export ---')
    try:
        dummy_trace = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
        traced = torch.jit.trace(export_model, dummy_trace)
        torch.onnx.export(
            traced,
            dummy_trace,
            str(BEST_ONNX_PATH),
            input_names=['images'],
            output_names=['logits'],
            dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=17,
            do_constant_folding=True,
        )
        print('✅ Exportado via jit.trace')
        export_success = True
    except Exception as e3:
        print(f'Falló: {str(e3)[:200]}')

# Estrategia 4: export directo sin dynamic_axes
if not export_success:
    print('\n--- Estrategia 4: export directo ---')
    try:
        torch.onnx.export(
            export_model,
            dummy_input,
            str(BEST_ONNX_PATH),
            input_names=['images'],
            output_names=['logits'],
            dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=18,
            do_constant_folding=True,
            export_params=True,
        )
        print('⚠️ Exportado (puede tener limitaciones de batch)')
        export_success = True
    except Exception as e4:
        print(f'Falló: {str(e4)[:200]}')

assert export_success, 'No se pudo exportar ONNX con ninguna estrategia'


# ══════════════════════════════════════════════════════════════════════════════
# Verificar ONNX
# ══════════════════════════════════════════════════════════════════════════════

print('\n=== Verificación ONNX ===')

# Check model validity
onnx_model = onnx.load(str(BEST_ONNX_PATH))
onnx.checker.check_model(onnx_model)
onnx_size_mb = BEST_ONNX_PATH.stat().st_size / 1024 / 1024
print(f'ONNX checker OK | {onnx_size_mb:.1f} MB')
print(f'IR version: {onnx_model.ir_version} | '
      f'Opset: {onnx_model.opset_import[0].version}')

# Create ONNX Runtime session
sess = ort.InferenceSession(str(BEST_ONNX_PATH),
                             providers=['CPUExecutionProvider'])

# Test batch=1
ort_out1 = sess.run(None, {'images': dummy_input.numpy()})[0]
print(f'\nBatch=1: shape={ort_out1.shape} '
      f'range=[{ort_out1.min():.1f}, {ort_out1.max():.1f}]')

# ✅ FIX: Verificar escala con abs().max() > 1.0
#    Coseno crudo ∈ [-1, 1], así que si abs().max() > 1.0 la escala s se aplicó
assert ort_out1.abs().max() > 1.0, \
    f'Escala ArcFace no incluida: abs max={ort_out1.abs().max():.2f} ' \
    f'(debería ser >> 1.0 con s={ARCFACE_S})'
print(f'✅ Escala ArcFace OK (abs max={ort_out1.abs().max():.1f}, '
      f'confirma s={ARCFACE_S} aplicado)')

# Test batch=4
try:
    dummy4 = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).numpy()
    ort_out4 = sess.run(None, {'images': dummy4})[0]
    print(f'Batch=4: shape={ort_out4.shape} ✅ Dynamic batch funciona')
except Exception as e:
    print(f'Batch=4: solo batch=1 (OK para inferencia)')


# ══════════════════════════════════════════════════════════════════════════════
# Verificación JUSTA: mismas imágenes PyTorch vs ONNX
# ══════════════════════════════════════════════════════════════════════════════

print('\n=== Verificación: PyTorch vs ONNX en imágenes de test ===')

# Test con imágenes reales del test set
model_cpu.eval()
match_count, total_count = 0, 0
diff_max = 0.0

for images, labels in test_loader:
    for i in range(min(len(images), 50)):  # hasta 50 por batch
        img_tensor = images[i:i+1]
        
        with torch.no_grad():
            pt_logits = model_cpu(img_tensor)
            pt_pred = pt_logits.argmax(1).item()
        
        ort_logits = sess.run(None, {'images': img_tensor.numpy()})[0]
        ort_pred = ort_logits.argmax(axis=1)[0]
        
        # Track max diff
        wrap_logits = export_model(img_tensor).detach().numpy()
        d = np.abs(ort_logits - wrap_logits).max()
        diff_max = max(diff_max, d)
        
        if pt_pred == ort_pred:
            match_count += 1
        total_count += 1
    
    if total_count >= 500:
        break

agreement = match_count / total_count
print(f'Acuerdo PyTorch↔ONNX: {match_count}/{total_count} = {100*agreement:.1f}%')
print(f'Max logit diff: {diff_max:.6f}')

if agreement >= 0.99:
    print('✅ ONNX validado — acuerdo ≥99%')
elif agreement >= 0.95:
    print('⚠️ ONNX aceptable — acuerdo ≥95% (diferencias numéricas menores)')
else:
    print('❌ ONNX tiene discrepancias significativas — revisar')


# ══════════════════════════════════════════════════════════════════════════════
# Test ONNX en caracteres específicos (acentuadas + confusos)
# ══════════════════════════════════════════════════════════════════════════════

print('\n=== Test ONNX: caracteres específicos ===')

test_chars_specific = {
    'Acentuadas': ['á', 'é', 'í', 'ó', 'ú', 'ñ'],
    'Bases':      ['a', 'e', 'i', 'o', 'u', 'n'],
    'Case':       ['c', 'C', 's', 'S'],
    'Shape':      ['1', 'l', '0', 'O'],
}

df_test_reset = df_test.reset_index(drop=True)

for group_name, chars in test_chars_specific.items():
    print(f'\n  {group_name}:')
    for ch in chars:
        ch_idx = char_to_idx(ch)
        if ch_idx is None:
            continue
        
        matching = df_test_reset[df_test_reset['class_idx'] == ch_idx].index.tolist()
        if not matching:
            print(f'    {ch!r}: sin datos en test')
            continue
        
        # Tomar una muestra
        ds_i = matching[0]
        tensor, _ = test_ds[ds_i]
        img_np = tensor.unsqueeze(0).numpy()
        
        # PyTorch
        with torch.no_grad():
            pt_logits = model_cpu(tensor.unsqueeze(0))
            pt_probs = torch.softmax(pt_logits, dim=1).squeeze()
            pt_pred = int(pt_probs.argmax())
            pt_conf = float(pt_probs[pt_pred])
        
        # ONNX
        ort_logits = sess.run(None, {'images': img_np})[0]
        ort_probs = np.exp(ort_logits) / np.exp(ort_logits).sum(axis=1, keepdims=True)
        ort_pred = int(ort_probs.argmax(axis=1)[0])
        ort_conf = float(ort_probs[0, ort_pred])
        
        match_tag = '✅' if pt_pred == ort_pred else '❌'
        correct_tag = '✓' if pt_pred == ch_idx else '✗'
        source = df_test_reset.iloc[ds_i]['source']
        
        print(f'    {ch!r}: PT→{IDX2CHAR[pt_pred]!r}({pt_conf:.2f}) '
              f'ONNX→{IDX2CHAR[ort_pred]!r}({ort_conf:.2f}) '
              f'{match_tag} correct:{correct_tag} [{source}]')


# ══════════════════════════════════════════════════════════════════════════════
# Resumen final
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n{"="*50}')
print(f'=== ONNX Export Resumen ===')
print(f'{"="*50}')
print(f'  Archivo:    {BEST_ONNX_PATH.name}')
print(f'  Tamaño:     {onnx_size_mb:.1f} MB')
print(f'  Opset:      {onnx_model.opset_import[0].version}')
print(f'  Acuerdo:    {100*agreement:.1f}%')
print(f'  Includes:   backbone + projector({EMBED_DIM}) + ArcFace(s={ARCFACE_S})')
print(f'  Input:      images [{1}, 3, {IMG_SIZE}, {IMG_SIZE}]')
print(f'  Output:     logits [{1}, {NUM_CLASSES}]')
print(f'{"="*50}')

# Cleanup
del model_cpu, export_model

## Cell 21 — Phase C: Función de inferencia

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 21 — Phase C: Función de inferencia actualizada (con projection head)
# ══════════════════════════════════════════════════════════════════════════════

_classify_char_cache: Dict = {}


def _load_model_from_checkpoint(model_path: str) -> Dict:
    """Carga modelo desde checkpoint con soporte para v4 (sin projector) y v5 (con projector)."""
    ckpt = torch.load(model_path, map_location='cpu')
    
    num_classes = ckpt.get('num_classes', NUM_CLASSES)
    embed_dim = ckpt.get('embed_dim', None)
    has_projector = ckpt.get('projection_head', False)
    img_size = ckpt.get('img_size', IMG_SIZE)
    idx2char = ckpt.get('char_map', IDX2CHAR)
    
    # Detectar automáticamente si el checkpoint tiene projector
    # mirando las keys del state_dict
    state_keys = set(ckpt['model_state_dict'].keys())
    has_projector_keys = any('projector' in k for k in state_keys)
    
    if has_projector_keys and embed_dim is None:
        # Inferir embed_dim del state_dict
        for k, v in ckpt['model_state_dict'].items():
            if 'projector.0.weight' in k:  # Linear layer
                embed_dim = v.shape[0]
                break
        if embed_dim is None:
            embed_dim = 512  # fallback
        has_projector = True
    
    if has_projector and embed_dim:
        # v5: modelo con projection head
        m = OCRClassifier(
            num_classes=num_classes,
            embed_dim=embed_dim
        )
    else:
        # v4 fallback: modelo sin projection head
        # Crear con la arquitectura antigua
        embed_dim = None
        m = OCRClassifier(num_classes=num_classes)
    
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    
    return {
        'model': m,
        'idx2char': idx2char,
        'img_size': img_size,
        'embed_dim': embed_dim,
        'has_projector': has_projector,
        'arcface_s': ckpt.get('arcface_s', ARCFACE_S),
    }


def classify_char(image: np.ndarray, model_path: str,
                  use_tta: bool = False) -> dict:
    """
    Clasifica un crop de carácter individual.

    Args:
        image:      array BGR o grayscale numpy (cualquier tamaño)
        model_path: ruta a best_classifier.pt
        use_tta:    si True, usa Test-Time Augmentation (más lento pero preciso)

    Returns:
        {
            'char': str,
            'class_idx': int,
            'confidence': float,
            'top5': [(char, prob), ...],
            'char_type': str,  # 'lowercase', 'uppercase', 'digit', 'accented', etc.
        }
    """
    global _classify_char_cache
    
    # Cargar modelo (con cache)
    if model_path not in _classify_char_cache:
        _classify_char_cache[model_path] = _load_model_from_checkpoint(model_path)
    
    cache = _classify_char_cache[model_path]
    m = cache['model']
    idx2char = cache['idx2char']
    sz = cache['img_size']
    
    # ── Preprocesar imagen ───────────────────────────────────────────────
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    img_lb = letterbox_resize(image, sz)
    img_rgb = cv2.cvtColor(img_lb, cv2.COLOR_BGR2RGB)
    
    # ── Inferencia ───────────────────────────────────────────────────────
    if use_tta and len(tta_transforms) > 1:
        # TTA: promediar probabilidades de múltiples augmentaciones
        avg_probs = np.zeros(NUM_CLASSES, dtype=np.float32)
        
        for tfm in tta_transforms:
            tensor = tfm(image=img_rgb)['image'].unsqueeze(0)
            with torch.no_grad():
                logits = m(tensor)
                probs = torch.softmax(logits, dim=1).squeeze().numpy()
            avg_probs += probs
        
        avg_probs /= len(tta_transforms)
        probs_final = torch.tensor(avg_probs)
    else:
        # Sin TTA: inferencia directa
        tfm = A.Compose([
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
        tensor = tfm(image=img_rgb)['image'].unsqueeze(0)
        
        with torch.no_grad():
            logits = m(tensor)
            probs_final = torch.softmax(logits, dim=1).squeeze()
    
    # ── Extraer resultados ───────────────────────────────────────────────
    top5_vals, top5_idxs = probs_final.topk(5)
    pred_idx = int(top5_idxs[0])
    confidence = float(top5_vals[0])
    pred_char = idx2char[pred_idx]
    
    # ── Clasificar tipo de carácter ──────────────────────────────────────
    char_type = _get_char_type(pred_char)
    
    return {
        'char': pred_char,
        'class_idx': pred_idx,
        'confidence': confidence,
        'char_type': char_type,
        'top5': [(idx2char[int(i)], float(v))
                 for i, v in zip(top5_idxs, top5_vals)],
    }


def _get_char_type(ch: str) -> str:
    """Clasifica el tipo de un carácter."""
    accented = set('áéíóúüÁÉÍÓÚÜñÑ')
    if ch in accented:
        return 'accented'
    if ch.isalpha() and ch.islower():
        return 'lowercase'
    if ch.isalpha() and ch.isupper():
        return 'uppercase'
    if ch.isdigit():
        return 'digit'
    if ch.startswith('línea') or ch in ('curva', 'círculo'):
        return 'stroke'
    return 'punctuation'


def classify_char_onnx(image: np.ndarray, onnx_path: str,
                       char_map: Dict[int, str]) -> dict:
    """
    Inferencia con ONNX Runtime (más rápido, portable).
    
    Args:
        image:     array BGR o grayscale numpy
        onnx_path: ruta a best_classifier.onnx
        char_map:  dict idx→char
    
    Returns:
        mismo formato que classify_char()
    """
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    img_lb = letterbox_resize(image, IMG_SIZE)
    img_rgb = cv2.cvtColor(img_lb, cv2.COLOR_BGR2RGB)
    
    tfm = A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    tensor = tfm(image=img_rgb)['image'].unsqueeze(0).numpy()
    
    # ONNX inference
    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    logits = session.run(None, {'images': tensor})[0]
    
    # Softmax
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = (exp_logits / exp_logits.sum(axis=1, keepdims=True)).squeeze()
    
    top5_idx = probs.argsort()[-5:][::-1]
    pred_idx = int(top5_idx[0])
    confidence = float(probs[pred_idx])
    pred_char = char_map[pred_idx]
    
    return {
        'char': pred_char,
        'class_idx': pred_idx,
        'confidence': confidence,
        'char_type': _get_char_type(pred_char),
        'top5': [(char_map[int(i)], float(probs[i])) for i in top5_idx],
    }


# ══════════════════════════════════════════════════════════════════════════════
# Smoke tests
# ══════════════════════════════════════════════════════════════════════════════

print('=== Smoke Tests ===')

# Test 1: classify_char con PyTorch (sin TTA)
test_img_row = df_test.iloc[0]
test_img_bgr = cv2.imread(test_img_row['image_path'])
if test_img_bgr is not None:
    res = classify_char(test_img_bgr, str(BEST_PT_PATH), use_tta=False)
    print(f'\n1. classify_char (sin TTA):')
    print(f'   Resultado: {res["char"]!r} (conf={res["confidence"]:.3f}, '
          f'type={res["char_type"]})')
    print(f'   Top5: {res["top5"]}')
    assert res['char'] in CHAR2IDX
    assert 0.0 <= res['confidence'] <= 1.0
    assert len(res['top5']) == 5
    print(f'   ✅ OK')

# Test 2: classify_char con TTA
if test_img_bgr is not None:
    res_tta = classify_char(test_img_bgr, str(BEST_PT_PATH), use_tta=True)
    print(f'\n2. classify_char (con TTA):')
    print(f'   Resultado: {res_tta["char"]!r} (conf={res_tta["confidence"]:.3f})')
    match = '✅' if res_tta['char'] == res['char'] else '⚠️ diferente'
    print(f'   vs sin TTA: {match}')

# Test 3: classify_char_onnx
if test_img_bgr is not None and BEST_ONNX_PATH.exists():
    res_onnx = classify_char_onnx(test_img_bgr, str(BEST_ONNX_PATH), IDX2CHAR)
    print(f'\n3. classify_char_onnx:')
    print(f'   Resultado: {res_onnx["char"]!r} (conf={res_onnx["confidence"]:.3f})')
    match = '✅' if res_onnx['char'] == res['char'] else '⚠️ diferente'
    print(f'   vs PyTorch: {match}')

# Test 4: verificar con caracteres acentuados del test set
print(f'\n4. Test de acentuadas:')
df_test_reset = df_test.reset_index(drop=True)
for ch in ['á', 'é', 'ñ', 'a', 'e', 'n']:
    ch_idx = char_to_idx(ch)
    if ch_idx is None:
        continue
    matching = df_test_reset[df_test_reset['class_idx'] == ch_idx].index.tolist()
    if not matching:
        print(f'   {ch!r}: sin datos en test')
        continue
    
    ds_i = matching[0]
    img_path = df_test_reset.iloc[ds_i]['image_path']
    source = df_test_reset.iloc[ds_i]['source']
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    
    res = classify_char(img_bgr, str(BEST_PT_PATH), use_tta=False)
    correct = '✅' if res['char'] == ch else '❌'
    print(f'   {ch!r} [{source}]: pred={res["char"]!r} '
          f'conf={res["confidence"]:.2f} {correct}')

print(f'\n✅ Todas las funciones de inferencia operativas')

## Cell 22 — Phase C: metrics_report.json

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 22 — Phase C: metrics_report.json completo
# ══════════════════════════════════════════════════════════════════════════════

# ── Compilar estadísticas por tipo de clase ──────────────────────────────────
real_class_stats = {}
accent_class_stats = {}
synth_class_stats = {}

for idx in range(NUM_CLASSES):
    ch = IDX2CHAR[idx]
    acc = per_class_acc.get(ch, 0.0)
    f1 = per_class_f1.get(ch, 0.0)
    dtype = class_data_type.get(idx, 'unknown')
    entry = {'accuracy': round(acc, 4), 'f1': round(f1, 4)}
    
    if dtype == 'real':
        real_class_stats[ch] = entry
    elif dtype == 'accent_aug':
        accent_class_stats[ch] = entry
    elif dtype == 'synthetic':
        synth_class_stats[ch] = entry

# ── Construir reporte ────────────────────────────────────────────────────────
metrics_report = {
    'run_id': RUN_ID,
    'model': 'tf_efficientnetv2_s + ProjectionHead + ArcFace v5',
    'architecture': {
        'backbone': 'tf_efficientnetv2_s',
        'backbone_features': 1280,
        'projection_head': f'Linear(1280→{EMBED_DIM}) + BN + ReLU + Dropout({DROPOUT_RATE})',
        'embed_dim': EMBED_DIM,
        'arcface': f'ArcFace(s={ARCFACE_S}, m={ARCFACE_M})',
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
    },
    
    # ── Datos ────────────────────────────────────────────────────────────
    'data': {
        'total_train_images': len(train_ds),
        'total_val_images': len(val_ds),
        'total_test_images': len(test_ds),
        'n_real_classes': len(real_classes),
        'n_accent_aug_classes': len(accent_classes),
        'n_synth_only_classes': len(synth_classes),
        'accent_samples_per_base': ACCENT_SAMPLES_PER_BASE,
        'synth_per_class': SYNTH_PER_CLASS,
        'emnist_max_per_class': EMNIST_MAX_PER_CLASS,
        'test_strategy': 'real/accent_aug data in test for real classes; '
                        'synthetic in test only for synth-only classes',
    },
    
    # ── Entrenamiento ────────────────────────────────────────────────────
    'training': {
        'best_epoch': best_epoch,
        'total_epochs': len(history['train_loss']),
        'training_time_minutes': round(training_time_min, 2),
        'freeze_epochs': FREEZE_EPOCHS,
        'warmup_epochs': 3,
        'lr_head': LR_HEAD,
        'lr_backbone': LR_BACKBONE,
        'weight_decay': WEIGHT_DECAY,
        'dropout_rate': DROPOUT_RATE,
        'label_smoothing': LABEL_SMOOTH,
        'mixup_alpha': MIXUP_ALPHA,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR (sin restarts)',
        'loss': 'FocalLoss(gamma=2.0) + class_weights + accent_boost',
        'accent_boost_in_loss': ACCENT_BOOST,
    },
    
    # ── Métricas globales ────────────────────────────────────────────────
    'metrics_global': {
        'best_val_acc': round(float(best_val_acc), 4),
        'weighted_f1': round(float(val_f1), 4),
        'test_acc': round(float(test_acc), 4),
        'tta_n': TTA_N,
    },
    
    # ── Métricas honestas (LAS MÁS IMPORTANTES) ─────────────────────────
    'metrics_honest': {
        'real_test_acc': round(float(real_test_acc), 4),
        'accent_test_acc': round(float(accent_test_acc), 4),
        'synth_test_acc': round(float(synth_test_acc), 4),
        'note': 'real_test_acc es el mejor predictor del rendimiento en '
                'datos reales ("carpet test"). synth_test_acc está inflado.',
    },
    
    # ── Métricas por tipo de clase ───────────────────────────────────────
    'per_type_stats': {
        'real_classes': {
            'count': len(real_class_stats),
            'mean_acc': round(np.mean([s['accuracy'] for s in real_class_stats.values()]), 4)
                        if real_class_stats else 0,
            'mean_f1': round(np.mean([s['f1'] for s in real_class_stats.values()]), 4)
                       if real_class_stats else 0,
            'classes': real_class_stats,
        },
        'accent_aug_classes': {
            'count': len(accent_class_stats),
            'mean_acc': round(np.mean([s['accuracy'] for s in accent_class_stats.values()]), 4)
                        if accent_class_stats else 0,
            'mean_f1': round(np.mean([s['f1'] for s in accent_class_stats.values()]), 4)
                       if accent_class_stats else 0,
            'classes': accent_class_stats,
        },
        'synth_only_classes': {
            'count': len(synth_class_stats),
            'mean_acc': round(np.mean([s['accuracy'] for s in synth_class_stats.values()]), 4)
                        if synth_class_stats else 0,
            'mean_f1': round(np.mean([s['f1'] for s in synth_class_stats.values()]), 4)
                       if synth_class_stats else 0,
            'classes': synth_class_stats,
        },
    },
    
    # ── Accent confusion ─────────────────────────────────────────────────
    'accent_confusion': accent_confusion_details if accent_confusion_details else [],
    
    # ── Pares confusos ───────────────────────────────────────────────────
    'top_confused_pairs': top10_pairs,
    
    # ── Clases problemáticas ─────────────────────────────────────────────
    'problematic_classes': {
        'f1_below_50': classes_below_50,
        'f1_below_80': classes_below_80,
        'n_below_50': len(classes_below_50),
        'n_below_80': len(classes_below_80),
    },
    
    # ── Per-class metrics completas ──────────────────────────────────────
    'per_class_f1': per_class_f1,
    'per_class_acc': per_class_acc,
    
    # ── Cambios vs versión anterior ──────────────────────────────────────
    'improvements_v5': [
        'Projection Head 1280→512 con BN+ReLU+Dropout',
        'Accent Augmentation: acentos dibujados sobre EMNIST real',
        f'  → {ACCENT_SAMPLES_PER_BASE} imgs por clase acentuada',
        'Test honesto: datos reales en test para clases reales',
        'FocalLoss con class_weights (effective number of samples)',
        f'  → Accent classes boosted {ACCENT_BOOST}x en loss',
        'Source-weighted sampling (verack 1.5x, accent_aug 1.3x, synthetic 0.7x)',
        f'EMNIST_MAX_PER_CLASS: 500→{EMNIST_MAX_PER_CLASS}',
        f'SYNTH_PER_CLASS: 100→{SYNTH_PER_CLASS}',
        f'FREEZE_EPOCHS: 2→{FREEZE_EPOCHS} (estabilizar projector+ArcFace)',
        'Warmup 3 epochs con LR lineal 10%→100%',
        'CosineAnnealingLR sin restarts (más estable que WarmRestarts)',
        f'LR_HEAD: 1e-2→{LR_HEAD}',
        f'LR_BACKBONE: 2e-4→{LR_BACKBONE}',
        f'DROPOUT: 0.5→{DROPOUT_RATE}',
        'Backbone unfreeze gradual: early×0.1, mid×0.3, late×1.0',
        'ElasticTransform más fuerte (alpha=1.5, p=0.4)',
        'Morphological ops para simular grosor de trazo',
        f'TTA: 4→{TTA_N} transforms (añade elastic)',
        'Hard pairs ampliados: +28 pares acentuados (a/á, e/é, n/ñ...)',
        'classify_char() soporta TTA y auto-detecta projection head',
        'classify_char_onnx() nueva función para inferencia ONNX',
    ],
    
    'seed': SEED,
}

# ── Guardar ──────────────────────────────────────────────────────────────────
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_report, f, indent=2, ensure_ascii=False)

print('Guardado metrics_report.json')

# ── Resumen en consola ───────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'=== RESUMEN FINAL v5 ===')
print(f'{"="*60}')
print(f'  Métricas globales:')
print(f'    best_val_acc:       {best_val_acc:.4f}')
print(f'    weighted_f1:        {val_f1:.4f}')
print(f'    test_acc:           {test_acc:.4f}')
print(f'')
print(f'  Métricas honestas:')
print(f'    real_test_acc:      {real_test_acc:.4f} ← predictor de "carpet"')
print(f'    accent_test_acc:    {accent_test_acc:.4f} ← mejora principal')
print(f'    synth_test_acc:     {synth_test_acc:.4f} ← inflado')
print(f'')
print(f'  Clases problemáticas:')
print(f'    F1 < 0.5: {len(classes_below_50):3d} clases')
print(f'    F1 < 0.8: {len(classes_below_80):3d} clases')
print(f'')
print(f'  Por tipo:')
if real_class_stats:
    r_mean = np.mean([s['accuracy'] for s in real_class_stats.values()])
    print(f'    Real ({len(real_class_stats):2d} cls):       mean_acc={r_mean:.4f}')
if accent_class_stats:
    a_mean = np.mean([s['accuracy'] for s in accent_class_stats.values()])
    print(f'    Accent ({len(accent_class_stats):2d} cls):     mean_acc={a_mean:.4f}')
if synth_class_stats:
    s_mean = np.mean([s['accuracy'] for s in synth_class_stats.values()])
    print(f'    Synthetic ({len(synth_class_stats):2d} cls):  mean_acc={s_mean:.4f} (inflado)')
print(f'{"="*60}')

## Cell 23 — Resumen de artefactos

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 23 — Resumen de artefactos (actualizado con nuevos archivos)
# ══════════════════════════════════════════════════════════════════════════════

artifacts = [
    # Datos y configuración
    (META_PATH,                              'merged_metadata.parquet'),
    (COVERAGE_PATH,                          'class_coverage_report.json'),
    (TRAIN_CFG_PATH,                         'train_config.json'),
    
    # Modelo
    (BEST_PT_PATH,                           'best_classifier.pt'),
    (BEST_ONNX_PATH,                         'best_classifier.onnx'),
    
    # Métricas
    (METRICS_PATH,                           'metrics_report.json'),
    (TOP10_PATH,                             'top10_confused_pairs.json'),
    
    # Visualizaciones de entrenamiento
    (CURVES_PATH,                            'training_curves.png'),
    
    # Confusion matrices
    (CONFMAT_PATH,                           'confusion_matrix.png'),
    (WORK / 'accent_confusion_detail.png',   'accent_confusion_detail.png'),
    
    # Predicciones
    (WORK / 'directed_predictions.png',      'directed_predictions.png'),
    (WORK / 'base_vs_accent_comparison.png', 'base_vs_accent_comparison.png'),
    (WORK / 'sample_predictions.png',        'sample_predictions.png'),
    
    # Accuracy
    (WORK / 'per_class_accuracy.png',        'per_class_accuracy.png'),
    (WORK / 'per_class_accuracy_by_type.png', 'per_class_accuracy_by_type.png'),
    
    # Accent augmentation QC
    (WORK / 'accent_augmentation_samples.png', 'accent_augmentation_samples.png'),
]

print(f'=== Resumen de artefactos v5 ===')
print(f'{"Archivo":<50s} {"Tamaño":>10s} {"Status":>6s}')
print(f'{"-"*68}')

total_size = 0
for path, name in artifacts:
    p = Path(path)
    if p.exists():
        size_kb = p.stat().st_size / 1024
        total_size += size_kb
        status = '✓'
        size_str = f'{size_kb:8.1f} KB'
    else:
        status = '✗'
        size_str = 'FALTANTE'
    print(f'  {status} {name:<48s} {size_str}')

print(f'{"-"*68}')
print(f'  Total: {total_size/1024:.1f} MB')

# ── Resumen del run ──────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'=== Run ID: {RUN_ID} ===')
print(f'{"="*60}')
print(f'  best_val_acc:    {best_val_acc:.4f}')
print(f'  test_acc:        {test_acc:.4f} (global, TTA×{TTA_N})')
print(f'  real_test_acc:   {real_test_acc:.4f} (honesto)')
print(f'  accent_test_acc: {accent_test_acc:.4f}')
print(f'  best_epoch:      {best_epoch}')
print(f'  training_time:   {training_time_min:.0f} min')
print(f'{"="*60}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 24 — Descargar todos los artefactos en un ZIP
# ══════════════════════════════════════════════════════════════════════════════
import zipfile

ZIP_PATH = WORK / f'classifier_v5_{RUN_ID}.zip'

# Lista de artefactos a incluir en el ZIP
artifacts_to_zip = [
    # Datos y configuración
    (META_PATH,                              'merged_metadata.parquet'),
    (COVERAGE_PATH,                          'class_coverage_report.json'),
    (TRAIN_CFG_PATH,                         'train_config.json'),
    (CHAR_MAP_PATH,                          'char_map.json'),
    
    # Modelo
    (BEST_PT_PATH,                           'best_classifier.pt'),
    (BEST_ONNX_PATH,                         'best_classifier.onnx'),
    
    # Métricas
    (METRICS_PATH,                           'metrics_report.json'),
    (TOP10_PATH,                             'top10_confused_pairs.json'),
    
    # Visualizaciones
    (CURVES_PATH,                            'training_curves.png'),
    (CONFMAT_PATH,                           'confusion_matrix.png'),
    (WORK / 'accent_confusion_detail.png',   'accent_confusion_detail.png'),
    (WORK / 'directed_predictions.png',      'directed_predictions.png'),
    (WORK / 'base_vs_accent_comparison.png', 'base_vs_accent_comparison.png'),
    (WORK / 'sample_predictions.png',        'sample_predictions.png'),
    (WORK / 'per_class_accuracy.png',        'per_class_accuracy.png'),
    (WORK / 'per_class_accuracy_by_type.png', 'per_class_accuracy_by_type.png'),
    (WORK / 'accent_augmentation_samples.png', 'accent_augmentation_samples.png'),
]

print(f'Creando ZIP: {ZIP_PATH.name}')
print(f'{"Archivo":<50s} {"Tamaño":>10s} {"Status":>6s}')
print(f'{"-"*68}')

total_zipped = 0
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_path, arc_name in artifacts_to_zip:
        src = Path(src_path)
        if src.exists():
            zf.write(src, arc_name)
            size_kb = src.stat().st_size / 1024
            total_zipped += size_kb
            print(f'  ✅ {arc_name:<48s} {size_kb:8.1f} KB')
        else:
            print(f'  ⚠️  {arc_name:<48s} NO ENCONTRADO')

zip_size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
print(f'{"-"*68}')
print(f'\n📦 ZIP creado: {ZIP_PATH.name} ({zip_size_mb:.1f} MB)')
print(f'   Contenido: {total_zipped/1024:.1f} MB sin comprimir')

# ── Resumen final completo ───────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'{"RESUMEN FINAL v5":^60}')
print(f'{"="*60}')
print(f'')
print(f'  📊 Métricas:')
print(f'     val_acc:          {best_val_acc:.4f}')
print(f'     test_acc:         {test_acc:.4f} (global)')
print(f'     real_test_acc:    {real_test_acc:.4f} ← predictor "carpet"')
print(f'     accent_test_acc:  {accent_test_acc:.4f}')
print(f'')
print(f'  🏗️ Arquitectura:')
print(f'     EfficientNetV2-S → Proj({EMBED_DIM}) → ArcFace({NUM_CLASSES})')
print(f'')
print(f'  📦 Artefactos:')
print(f'     {ZIP_PATH.name} ({zip_size_mb:.1f} MB)')
print(f'')
print(f'  🔑 Mejoras clave v5:')
print(f'     1. Accent Augmentation desde EMNIST real')
print(f'     2. Projection Head 1280→{EMBED_DIM}')
print(f'     3. FocalLoss + class weights + accent boost')
print(f'     4. Test honesto (datos reales en test)')
print(f'     5. Warmup + CosineAnnealing estable')
print(f'{"="*60}')

# ── Descargar automáticamente en Kaggle ──────────────────────────────────────
try:
    from IPython.display import FileLink, display
    display(FileLink(
        str(ZIP_PATH.relative_to(Path('/kaggle/working'))),
        result_html_prefix='📥 Descargar: '
    ))
except Exception:
    print(f'\n💡 Descarga manual: {ZIP_PATH}')
    print('   En Kaggle: Output → botón Download junto al archivo')